# 0. Final DSS Configuration

> **Repository execution note**
>
> Open the cloned repository as a folder in VS Code or Jupyter before running this notebook.
> The configuration cell detects the repository root from `requirements.txt`, creates the
> `data`, `outputs`, `reports`, `figures`, and `models` folders, and forces CPU-only execution.
>
> The TED 2022 annotated gold-standard file is intentionally not auto-copied into the active
> data folder. When Section 3 reports that the file is missing, manually copy or upload
> `assets/gold_standard_upload/ted2022_gold_standard_valid_979.csv` to
> `data/gold_standard/ted2022_gold_standard_valid_979.csv`, re-run Section 3, and then run
> the notebook from the top.


In [ ]:
# ================================================================
# 0. FINAL DSS CONFIGURATION — NO QUICK RUN — CPU ONLY
# ================================================================

from pathlib import Path
import os
import subprocess

# ------------------------------------------------
# Force CPU-only execution for the full pipeline
# ------------------------------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"


# ------------------------------------------------
# Detect a writable repository root
# ------------------------------------------------

def detect_project_root():
    """
    Prefer the cloned repository root, identified by requirements.txt.
    If the notebook is opened outside the repository workspace, fall back
    to a writable user-level directory.
    """
    cwd = Path.cwd().resolve()

    candidates = [cwd]
    candidates.extend(list(cwd.parents)[:3])

    for candidate in candidates:
        if (candidate / "requirements.txt").exists():
            return candidate

    fallback = Path.home() / "Deniz_thesis_pipeline"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


PROJECT_ROOT = detect_project_root()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
EXTRACTED_DIR = RAW_DIR / "ted2023_extracted"
GOLD_DIR = DATA_DIR / "gold_standard"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for p in [
    DATA_DIR,
    RAW_DIR,
    EXTRACTED_DIR,
    GOLD_DIR,
    PROCESSED_DIR,
    OUTPUTS_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    MODELS_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# Confirm that the selected location is writable.
write_test_path = PROJECT_ROOT / "_write_permission_test.txt"
write_test_path.write_text("write permission confirmed", encoding="utf-8")
write_test_path.unlink()


# ------------------------------------------------
# GPU is intentionally disabled
# ------------------------------------------------

GPU_AVAILABLE = False
GPU_NAME = None


CONFIG = {
    "seed": 42,
    "full_data_mode": True,
    "raw_sampling_allowed": False,
    "ted2023_zip_url": "https://data.europa.eu/api/hub/store/data/ted-contract-award-notices-2023.zip",

    "compute": {
        "use_gpu_if_available": False,
        "gpu_available": False,
        "gpu_name": None,
        "xgboost_device": "cpu",
        "lightgbm_device_type": "cpu",
    },

    "candidate_generation": {
        "max_pairs_per_positive_group": 200,
        "negative_ratio": 1.0,
        "max_negative_attempts_multiplier": 20,
        "min_group_size_for_pairs": 2,
        "hard_negative_share": 0.70,
        "random_negative_share": 0.30,
    },

    "features": {
        "similarity_features": [
            "cae_levenshtein",
            "cae_jaccard",
            "cae_weighted_ratio",
            "win_levenshtein",
            "win_jaccard",
            "win_weighted_ratio",
            "title_levenshtein",
            "title_jaccard",
            "title_weighted_ratio",
        ],
        "structured_features": [
            "value_ratio",
            "value_abs_log_diff",
            "same_cpv",
            "same_country",
            "same_award_date",
            "same_lot",
        ],
        "identifier_like_features": [
            "same_contract_number",
            "same_award_id",
        ],
    },

    "models": {
        "validation_size": 0.20,
        "cv_folds": 3,
        "tuning_sample_size": 100000,
        "tuned_models": [
            "RandomForest",
            "XGBoost",
            "LightGBM",
        ],
        "optuna_trials_by_model": {
            "RandomForest": 15,
            "XGBoost": 25,
            "LightGBM": 25,
        },
        "optuna_timeout_per_model_seconds": None,
        "primary_metric": "f1",
        "threshold_grid": [
            round(x, 2)
            for x in [i / 100 for i in range(5, 96, 5)]
        ],
    },

    "bootstrap": {
        "n_bootstrap": 500,
        "ci_low": 2.5,
        "ci_high": 97.5,
    },

    "shap": {
        "max_background_rows": 1000,
        "max_explain_rows": 1500,
    },

    "synthetic": {
        "enabled": True,
        "ctgan_epochs": 50,
        "tvae_epochs": 50,
        "num_synthetic_rows": 100000,
        "memorisation_nn_thresholds": [
            0.95,
            0.98,
            0.99,
        ],
    },
}

RANDOM_SEED = CONFIG["seed"]

FEATURE_COLUMNS_FULL = (
    CONFIG["features"]["similarity_features"]
    + CONFIG["features"]["structured_features"]
    + CONFIG["features"]["identifier_like_features"]
)

FEATURE_COLUMNS_REDUCED = (
    CONFIG["features"]["similarity_features"]
    + CONFIG["features"]["structured_features"]
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FULL_DATA_MODE:", CONFIG["full_data_mode"])
print("RAW SAMPLING ALLOWED:", CONFIG["raw_sampling_allowed"])
print("CPU ONLY MODE:", True)
print("GPU AVAILABLE:", CONFIG["compute"]["gpu_available"])
print("GPU NAME:", CONFIG["compute"]["gpu_name"])
print("XGBoost device:", CONFIG["compute"]["xgboost_device"])
print("LightGBM device:", CONFIG["compute"]["lightgbm_device_type"])
print("Feature columns full:", len(FEATURE_COLUMNS_FULL))
print("Feature columns reduced:", len(FEATURE_COLUMNS_REDUCED))
print("Expected gold-standard path:", GOLD_DIR / "ted2022_gold_standard_valid_979.csv")


# 1. Dependency Bootstrap and Hard Dependency Check

This section makes the notebook self-contained for a cloned repository. It checks the
runtime environment and, only when packages are missing, installs the CPU build of
PyTorch and the packages listed in `requirements.txt` into the active Python kernel.
The following cell then verifies every required import and stops explicitly if a
dependency is still unavailable.

For the most reliable execution, use the provided setup script or `environment.yml`
before opening the notebook. The in-notebook bootstrap is a fallback that removes the
need for repeated manual `pip install` commands.


In [ ]:
# ================================================================
# 1A. INSTALL OR REPAIR DEPENDENCIES FROM requirements.txt
# ================================================================

import importlib.util
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"This repository requires Python 3.11. Active version: {sys.version.split()[0]}. "
        "Select the Python 3.11 (deniz_dss) kernel."
    )

REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"

MODULES_REQUIRED_FOR_RUN = [
    "numpy",
    "pandas",
    "pyarrow",
    "sklearn",
    "matplotlib",
    "seaborn",
    "rapidfuzz",
    "optuna",
    "xgboost",
    "lightgbm",
    "shap",
    "openpyxl",
    "sdv",
    "torch",
    "joblib",
]

missing_modules = [
    module_name
    for module_name in MODULES_REQUIRED_FOR_RUN
    if importlib.util.find_spec(module_name) is None
]

try:
    installed_numpy_version = version("numpy")
except PackageNotFoundError:
    installed_numpy_version = None

numpy_repair_required = installed_numpy_version != "1.26.4"
installation_required = bool(missing_modules or numpy_repair_required)

if installation_required:
    if not REQUIREMENTS_PATH.exists():
        raise FileNotFoundError(
            f"requirements.txt was not found at {REQUIREMENTS_PATH}. "
            "Open the cloned repository as a folder before running the notebook."
        )

    print("Missing modules:", missing_modules)
    print("Installed NumPy:", installed_numpy_version)
    print("Required NumPy: 1.26.4")
    print("Installing or repairing dependencies in:", sys.executable)

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ])

    # Install the CPU build explicitly before the complete requirements file.
    if "torch" in missing_modules:
        if platform.system() in {"Windows", "Linux"}:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "--index-url",
                "https://download.pytorch.org/whl/cpu",
                "torch>=2.4,<3.0",
            ])
        else:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "torch>=2.4,<3.0",
            ])

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REQUIREMENTS_PATH),
    ])

    print("Dependency installation or repair completed.")
    print(
        "If the next cell reports a binary import/version error, restart the "
        "kernel once and run all again."
    )
else:
    print("All runtime dependencies are already installed.")
    print("NumPy version:", installed_numpy_version)


In [ ]:
# ================================================================
# 1B. HARD DEPENDENCY CHECK — NO SILENT MODEL SKIPPING
# ================================================================

import importlib
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "numpy": "numpy==1.26.4",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "rapidfuzz": "rapidfuzz",
    "optuna": "optuna",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "shap": "shap",
    "openpyxl": "openpyxl",
    "sdv": "sdv",
    "torch": "torch",
    "joblib": "joblib",
}

missing = []
for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except Exception as e:
        missing.append((module_name, pip_name, str(e)))

if missing:
    details = "\n".join(
        f"- import {module_name} failed ({error}). Package: {pip_name}"
        for module_name, pip_name, error in missing
    )
    raise ImportError(
        "Required DSS-final packages are missing or incompatible.\n"
        + details
        + f"\n\nInstall from: {PROJECT_ROOT / 'requirements.txt'}"
    )

import numpy as _np

if _np.__version__ != "1.26.4":
    raise RuntimeError(
        f"NumPy {_np.__version__} is active, but this repository is locked to NumPy 1.26.4. "
        "Install requirements.txt, restart the kernel, and run again."
    )

print("All required packages are available.")
print("Python executable:", sys.executable)
print("NumPy version:", _np.__version__)
print("No optional model skipping will be used.")


# 2. Imports, Seeding, and Utility Setup

This cell imports all libraries used in the notebook and fixes the random seed for reproducibility. It imports packages for data processing, plotting, feature engineering, machine learning, hyperparameter tuning, interpretability, nearest-neighbour analysis, and synthetic data generation. It also creates a unique run identifier and a run-specific output directory. This makes each pipeline execution traceable and prevents results from different runs from being confused.

In [ ]:
# ================================================================
# 2. IMPORTS, SEEDING, AND UTILITY FUNCTIONS
# ================================================================
import re
import json
import math
import time
import zipfile
import hashlib
import random
import platform
import subprocess
from itertools import combinations
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rapidfuzz import fuzz
import optuna
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.neighbors import NearestNeighbors
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

try:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
except Exception as e:
    raise ImportError("SDV import failed. Install/upgrade with: pip install -U sdv") from e

RANDOM_SEED = CONFIG["seed"]
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUTS_DIR / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID:", RUN_ID)
print("RUN_DIR:", RUN_DIR)


# 3. Locate, Download, and Extract TED Data

This cell locates the TED 2023 Contract Award Notices dataset and the external TED 2022 annotated gold-standard file. If the TED 2023 ZIP file is not found locally, it is downloaded from the configured EU data source. The ZIP file is then extracted if needed, and the largest CSV file is selected as the main TED 2023 source file. The TED 2022 annotated file is located separately and is reserved for final external testing only.

In [ ]:
# ================================================================
# 3. LOCATE / DOWNLOAD / EXTRACT TED 2023 AND LOCATE TED 2022 GOLD TEST
# ================================================================
def locate_file_by_patterns(root: Path, patterns):
    for pattern in patterns:
        matches = list(root.rglob(pattern))
        if matches:
            # prefer largest file if multiple matches
            matches = sorted(matches, key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)
            return matches[0]
    return None


def download_file(url: str, dest: Path):
    import urllib.request
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)
    return dest


def locate_or_download_ted2023_zip():
    patterns = [
        "TED - Contract award notices 2023*.zip",
        "*Contract*award*notices*2023*.zip",
        "ted*contract*award*notices*2023*.zip",
        "*2023*.zip",
    ]
    z = locate_file_by_patterns(PROJECT_ROOT, patterns)
    if z is None:
        z = RAW_DIR / "ted_contract_award_notices_2023.zip"
        download_file(CONFIG["ted2023_zip_url"], z)
    print("TED 2023 ZIP:", z, "size MB:", round(z.stat().st_size / 1024**2, 2))
    return z


def extract_zip_if_needed(zip_path: Path, extract_dir: Path):
    csvs = list(extract_dir.rglob("*.csv"))
    if csvs:
        print("Already extracted. CSV count:", len(csvs))
        return csvs
    print("Extracting:", zip_path)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    csvs = list(extract_dir.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV found after extraction in {extract_dir}")
    print("Extracted CSV count:", len(csvs))
    return csvs


def locate_ted2023_csv(extract_dir: Path):
    csvs = list(extract_dir.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError("No TED 2023 CSV found. Run extraction first.")
    csvs = sorted(csvs, key=lambda p: p.stat().st_size, reverse=True)
    selected = csvs[0]
    print("Selected TED 2023 CSV:", selected, "size MB:", round(selected.stat().st_size / 1024**2, 2))
    return selected


def locate_gold_2022():
    """
    The annotated external test set must be placed manually in GOLD_DIR.
    A source copy is bundled under assets/gold_standard_upload, but the
    pipeline intentionally does not auto-copy it.
    """
    expected = GOLD_DIR / "ted2022_gold_standard_valid_979.csv"
    bundled_source = (
        PROJECT_ROOT
        / "assets"
        / "gold_standard_upload"
        / "ted2022_gold_standard_valid_979.csv"
    )

    if expected.exists():
        print("TED 2022 gold-standard file:", expected)
        return expected

    print("\nACTION REQUIRED: TED 2022 gold-standard file was not found.")
    print("Expected destination:", expected)
    print("Bundled source copy:", bundled_source)
    print(
        "Copy or upload the bundled CSV to the expected destination, "
        "re-run this section, and then run the notebook from the top."
    )

    raise FileNotFoundError(
        "TED 2022 annotated gold-standard file not found at:\n"
        f"{expected}"
    )

ted2023_zip = locate_or_download_ted2023_zip()
extract_zip_if_needed(ted2023_zip, EXTRACTED_DIR)
ted2023_csv = locate_ted2023_csv(EXTRACTED_DIR)
ted2022_gold_path = locate_gold_2022()


# 4. Load Full TED 2023 Source Population and Standardise Schema

This cell loads the full TED 2023 source population into memory and standardises the schema into a smaller set of project-specific variables. It maps potentially different column names from the raw TED file to consistent fields such as authority, supplier, title, CPV code, country, award value, award date, lot, award ID, and contract number. If no record identifier exists, a synthetic record ID is created. This standardised table becomes the basis for cleaning, EDA, candidate-pair construction, and feature engineering.

In [ ]:
# ================================================================
# 4. LOAD FULL TED 2023 SOURCE POPULATION AND STANDARDISE SCHEMA
# ================================================================
def pick_col(df, options, required=True):
    cols_lower = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in cols_lower:
            return cols_lower[opt.lower()]
    if required:
        raise KeyError(f"None of these columns found: {options}")
    return None

start = time.time()
raw = pd.read_csv(ted2023_csv, low_memory=False)
print("Loaded TED 2023 raw shape:", raw.shape, "time sec:", round(time.time() - start, 1))

COLUMN_MAP = {
    "record_id": pick_col(raw, ["ID_NOTICE", "ID_NOTICE_CAN", "NOTICE_ID", "ID", "ID_AWARD"], required=False),
    "authority": pick_col(raw, ["CAE_NAME", "CONTRACTING_AUTHORITY_NAME", "AUTHORITY_NAME"]),
    "supplier": pick_col(raw, ["WIN_NAME", "WINNER_NAME", "SUPPLIER_NAME"]),
    "title": pick_col(raw, ["TITLE", "CONTRACT_TITLE"], required=False),
    "cpv": pick_col(raw, ["CPV", "CPV_CODE"]),
    "value_euro": pick_col(raw, ["VALUE_EURO", "VALUE", "AWARD_VALUE_EURO"], required=False),
    "country": pick_col(raw, ["ISO_COUNTRY_CODE", "COUNTRY", "CAE_NUTS"], required=False),
    "award_date": pick_col(raw, ["DT_AWARD", "AWARD_DATE", "DATE_AWARD"], required=False),
    "lot": pick_col(raw, ["ID_LOT", "LOT_ID", "LOT_NUMBER"], required=False),
    "award_id": pick_col(raw, ["ID_AWARD", "AWARD_ID"], required=False),
    "contract_number": pick_col(raw, ["CONTRACT_NUMBER", "CONTRACT_NO", "CONTRACT_ID"], required=False),
}
print("COLUMN_MAP:")
print(json.dumps(COLUMN_MAP, indent=2, default=str))

std = pd.DataFrame(index=raw.index)
for target, source in COLUMN_MAP.items():
    if source is None:
        std[target] = np.nan
    else:
        std[target] = raw[source]

if std["record_id"].isna().all():
    std["record_id"] = np.arange(len(std)).astype(str)

print("Standardised shape:", std.shape)
print(std.head())


# 5. Cleaning and Normalisation Functions

This cell defines the cleaning functions used throughout the pipeline. Text fields are lowercased, stripped, and normalised; code-like fields such as CPV, country, lot, award ID, and contract number are cleaned separately; dates are parsed safely; and monetary values are converted to numeric format, including common European number formats. The cell then creates normalised versions of key columns and saves the full standardised TED 2023 table as a processed Parquet file.

In [ ]:
# ================================================================
# 5. CLEANING / NORMALISATION FUNCTIONS
# ================================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"[\u00a0\t\r\n]+", " ", x)
    x = re.sub(r"[^\w\s\.-]", " ", x, flags=re.UNICODE)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def compact_text(x):
    x = clean_text(x)
    x = re.sub(r"[\W_]+", "", x, flags=re.UNICODE)
    return x


def clean_code(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"\.0$", "", x)
    return x


def parse_date_safe(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT
    return pd.to_datetime(x, errors="coerce", dayfirst=True)


def value_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace(" ", "")
    # Handle common European decimal formats.
    if s.count(",") == 1 and s.count(".") > 1:
        s = s.replace(".", "").replace(",", ".")
    elif s.count(",") == 1 and s.count(".") == 0:
        s = s.replace(",", ".")
    else:
        s = s.replace(",", "")
    try:
        return float(s)
    except Exception:
        return np.nan


def add_normalised_columns(df):
    out = df.copy()
    out["authority_norm"] = out["authority"].map(clean_text)
    out["supplier_norm"] = out["supplier"].map(clean_text)
    out["title_norm"] = out["title"].map(clean_text)
    out["authority_compact"] = out["authority"].map(compact_text)
    out["supplier_compact"] = out["supplier"].map(compact_text)
    out["title_compact"] = out["title"].map(compact_text)
    out["cpv_norm"] = out["cpv"].map(clean_code)
    out["country_norm"] = out["country"].map(clean_code)
    out["lot_norm"] = out["lot"].map(clean_code)
    out["award_id_norm"] = out["award_id"].map(clean_code)
    out["contract_number_norm"] = out["contract_number"].map(clean_code)
    out["award_date_norm"] = out["award_date"].map(parse_date_safe).dt.strftime("%Y-%m-%d").fillna("")
    out["value_float"] = out["value_euro"].map(value_to_float)
    out["value_bucket"] = out["value_float"].round(0).fillna(-1).astype("int64").astype(str)
    return out

std = add_normalised_columns(std)
std.to_parquet(PROCESSED_DIR / "ted2023_full_standardised.parquet", index=False)
print("Full TED 2023 standardised saved:", PROCESSED_DIR / "ted2023_full_standardised.parquet")
print("Rows:", len(std))


# 6. Full-Data Exploratory Data Analysis

This cell performs exploratory data analysis on the full TED 2023 source population. It produces summary statistics, missing-value rates, country distributions, CPV distributions, value distributions, award-date coverage, and duplicate-risk indicators. It also saves thesis-ready CSV files and figures. This stage supports Sprint 2 and provides evidence about the structure, coverage, missingness, and duplicate-risk patterns in the source data.

In [ ]:
# ================================================================
# 6. FULL-DATA EDA — OUTPUTS FOR SPRINT 2 AND THESIS RESULTS
# ================================================================
def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

eda_summary = {
    "n_records": int(len(std)),
    "n_columns_raw": int(raw.shape[1]),
    "n_columns_standardised": int(std.shape[1]),
    "n_countries": int(std["country_norm"].replace("", np.nan).nunique()),
    "n_cpv": int(std["cpv_norm"].replace("", np.nan).nunique()),
    "n_authorities_norm": int(std["authority_norm"].replace("", np.nan).nunique()),
    "n_suppliers_norm": int(std["supplier_norm"].replace("", np.nan).nunique()),
}

missing = std[["authority", "supplier", "title", "cpv", "value_euro", "country", "award_date", "lot", "award_id", "contract_number"]].isna().mean().sort_values(ascending=False).reset_index()
missing.columns = ["column", "missing_rate"]
missing.to_csv(REPORTS_DIR / "eda_missing_values.csv", index=False)

country_counts = std["country_norm"].replace("", np.nan).value_counts().reset_index()
country_counts.columns = ["country", "n_records"]
country_counts.to_csv(REPORTS_DIR / "eda_country_distribution.csv", index=False)

cpv_counts = std["cpv_norm"].replace("", np.nan).value_counts().head(50).reset_index()
cpv_counts.columns = ["cpv", "n_records"]
cpv_counts.to_csv(REPORTS_DIR / "eda_top50_cpv_distribution.csv", index=False)

# Duplicate-risk indicators
strict_key_cols = ["authority_compact", "supplier_compact", "title_compact", "cpv_norm", "country_norm", "award_date_norm", "lot_norm", "contract_number_norm", "value_bucket"]
admin_key_cols = ["award_id_norm", "lot_norm", "contract_number_norm", "country_norm", "cpv_norm"]
strict_group_sizes = std.groupby(strict_key_cols, dropna=False).size().reset_index(name="group_size")
admin_group_sizes = std[std["award_id_norm"].ne("")].groupby(admin_key_cols, dropna=False).size().reset_index(name="group_size") if std["award_id_norm"].ne("").any() else pd.DataFrame(columns=admin_key_cols+["group_size"])

dup_risk = pd.DataFrame([
    {"indicator": "strict_exact_repeated_key_groups", "n_groups": int((strict_group_sizes["group_size"] >= 2).sum()), "n_records_in_groups": int(strict_group_sizes.loc[strict_group_sizes["group_size"] >= 2, "group_size"].sum())},
    {"indicator": "admin_identifier_repeated_groups", "n_groups": int((admin_group_sizes["group_size"] >= 2).sum()) if len(admin_group_sizes) else 0, "n_records_in_groups": int(admin_group_sizes.loc[admin_group_sizes["group_size"] >= 2, "group_size"].sum()) if len(admin_group_sizes) else 0},
])
dup_risk.to_csv(REPORTS_DIR / "eda_duplicate_risk_indicators.csv", index=False)

with open(REPORTS_DIR / "eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(eda_summary, f, indent=2)

print("EDA summary:")
print(json.dumps(eda_summary, indent=2))
print("Duplicate-risk indicators:")
print(dup_risk)

plt.figure()
sns.barplot(data=missing, x="missing_rate", y="column")
plt.title("Missing-value rate in key TED 2023 fields")
savefig("eda_missing_values.png")

plt.figure(figsize=(10, 6))
sns.barplot(data=country_counts.head(25), x="n_records", y="country")
plt.title("Top countries by TED 2023 record count")
savefig("eda_country_distribution_top25.png")

plt.figure(figsize=(10, 6))
sns.barplot(data=cpv_counts.head(25), x="n_records", y="cpv")
plt.title("Top CPV categories by TED 2023 record count")
savefig("eda_cpv_distribution_top25.png")

plt.figure()
values = std["value_float"].replace([np.inf, -np.inf], np.nan).dropna()
values = values[values > 0]
if len(values):
    sns.histplot(np.log10(values), bins=60)
    plt.xlabel("log10(VALUE_EURO)")
    plt.title("Award value distribution, log scale")
    savefig("eda_value_distribution_log10.png")

plt.figure(figsize=(10, 5))
dates = pd.to_datetime(std["award_date_norm"], errors="coerce")
monthly = dates.dropna().dt.to_period("M").value_counts().sort_index()
monthly.plot(kind="bar")
plt.title("Award-date coverage by month")
plt.ylabel("n_records")
savefig("eda_award_date_monthly.png")


# 7. Full-Source Candidate-Pair Construction for Training

This cell constructs supervised training candidate pairs from the full TED 2023 source population. Positive pairs are generated using rule-based weak supervision based on repeated exact business keys, award identifiers, contract-lot information, and authority-supplier-title-value combinations. Negative pairs are generated using hard-negative and random-negative sampling strategies. This avoids constructing all possible record pairs while still creating informative duplicate and non-duplicate examples for supervised learning.

In [ ]:
# ================================================================
# 7. FULL-SOURCE CANDIDATE-PAIR CONSTRUCTION FOR TRAINING
# ================================================================
def pair_key(i, j):
    return tuple(sorted((int(i), int(j))))


def sample_pairs_from_group(indices, max_pairs_per_group, rng):
    idx = list(indices)
    n = len(idx)
    if n < 2:
        return []
    total_possible = n * (n - 1) // 2
    if max_pairs_per_group is None or total_possible <= max_pairs_per_group:
        return [pair_key(a, b) for a, b in combinations(idx, 2)]
    pairs = set()
    attempts = 0
    max_attempts = max_pairs_per_group * 20
    while len(pairs) < max_pairs_per_group and attempts < max_attempts:
        a, b = rng.choice(idx, size=2, replace=False)
        pairs.add(pair_key(a, b))
        attempts += 1
    return list(pairs)


def build_positive_pairs(df):
    rng = np.random.default_rng(RANDOM_SEED)
    positive_pairs = {}
    max_per_group = CONFIG["candidate_generation"]["max_pairs_per_positive_group"]

    rules = []
    # Strict exact repeated key across the full source population.
    rules.append(("strict_exact_repeated_key", ["authority_compact", "supplier_compact", "title_compact", "cpv_norm", "country_norm", "award_date_norm", "lot_norm", "contract_number_norm", "value_bucket"], None))
    # Same award identifier; high-precision weak supervision but leakage-prone if identifier features are used.
    if df["award_id_norm"].ne("").any():
        rules.append(("same_award_identifier", ["award_id_norm", "lot_norm", "country_norm", "cpv_norm"], lambda x: x["award_id_norm"].ne("")))
    # Same contract-lot within same country/CPV/date/supplier.
    if df["contract_number_norm"].ne("").any():
        rules.append(("same_contract_lot_supplier_date", ["contract_number_norm", "lot_norm", "supplier_compact", "country_norm", "cpv_norm", "award_date_norm"], lambda x: x["contract_number_norm"].ne("")))
    # Same authority/supplier/title/value within country and CPV.
    rules.append(("same_business_key_title_value", ["authority_compact", "supplier_compact", "title_compact", "country_norm", "cpv_norm", "value_bucket"], None))

    rule_rows = []
    for rule_name, group_cols, mask_fn in rules:
        source = df if mask_fn is None else df[mask_fn(df)]
        if source.empty:
            continue
        n_groups = 0
        n_pairs = 0
        for _, group in source.groupby(group_cols, dropna=False, sort=False):
            if len(group) < CONFIG["candidate_generation"]["min_group_size_for_pairs"]:
                continue
            pairs = sample_pairs_from_group(group.index.values, max_per_group, rng)
            if not pairs:
                continue
            n_groups += 1
            for p in pairs:
                if p not in positive_pairs:
                    positive_pairs[p] = rule_name
                    n_pairs += 1
        rule_rows.append({"rule": rule_name, "groups_used": n_groups, "pairs_added": n_pairs})
        print(rule_name, "groups:", n_groups, "new pairs:", n_pairs)

    pos = pd.DataFrame([{"idx1": a, "idx2": b, "label": 1, "source_rule": r} for (a, b), r in positive_pairs.items()])
    report = pd.DataFrame(rule_rows)
    return pos, report


def generate_negative_pairs(df, positive_pairs_set, target_n):
    rng = np.random.default_rng(RANDOM_SEED + 1)
    negatives = {}
    hard_target = int(target_n * CONFIG["candidate_generation"]["hard_negative_share"])
    random_target = target_n - hard_target

    def add_pair(a, b, rule):
        p = pair_key(a, b)
        if p in positive_pairs_set or p in negatives:
            return False
        negatives[p] = rule
        return True

    # Hard negatives: same supplier/CPV/country but different award/contract/title/date.
    hard_group_rules = [
        ("hard_negative_same_supplier_cpv_country", ["supplier_compact", "cpv_norm", "country_norm"]),
        ("hard_negative_same_authority_cpv_country", ["authority_compact", "cpv_norm", "country_norm"]),
        ("hard_negative_same_cpv_country_date", ["cpv_norm", "country_norm", "award_date_norm"]),
    ]

    for rule_name, cols in hard_group_rules:
        if len(negatives) >= hard_target:
            break
        groups = [(k, g.index.values) for k, g in df.groupby(cols, dropna=False, sort=False) if len(g) >= 2]
        rng.shuffle(groups)
        for _, idx in groups:
            if len(negatives) >= hard_target:
                break
            # limited attempts per group to keep broad coverage
            for _ in range(min(10, len(idx) * 2)):
                a, b = rng.choice(idx, size=2, replace=False)
                # Require at least one administrative difference to avoid accidentally sampling duplicates.
                row_a, row_b = df.loc[a], df.loc[b]
                if (
                    row_a["award_id_norm"] == row_b["award_id_norm"] and row_a["award_id_norm"] != ""
                ) or (
                    row_a["contract_number_norm"] == row_b["contract_number_norm"] and row_a["contract_number_norm"] != "" and row_a["lot_norm"] == row_b["lot_norm"]
                ):
                    continue
                add_pair(a, b, rule_name)
                if len(negatives) >= hard_target:
                    break

    # Random negatives from broad CPV-country blocks.
    broad_groups = [(k, g.index.values) for k, g in df.groupby(["cpv_norm", "country_norm"], dropna=False, sort=False) if len(g) >= 2]
    attempts = 0
    max_attempts = int(max(1000, random_target * CONFIG["candidate_generation"]["max_negative_attempts_multiplier"]))
    while len(negatives) < target_n and attempts < max_attempts and broad_groups:
        _, idx = broad_groups[rng.integers(0, len(broad_groups))]
        a, b = rng.choice(idx, size=2, replace=False)
        add_pair(a, b, "random_negative_same_cpv_country")
        attempts += 1

    neg = pd.DataFrame([{"idx1": a, "idx2": b, "label": 0, "source_rule": r} for (a, b), r in negatives.items()])
    return neg

start = time.time()
positive_pairs, positive_report = build_positive_pairs(std)
positive_report.to_csv(REPORTS_DIR / "candidate_positive_rule_report.csv", index=False)

if positive_pairs.empty:
    raise ValueError("No positive training pairs were constructed from full TED 2023. Review rules or data columns.")

target_neg = int(len(positive_pairs) * CONFIG["candidate_generation"]["negative_ratio"])
positive_pair_set = {pair_key(a, b) for a, b in positive_pairs[["idx1", "idx2"]].to_numpy()}
negative_pairs = generate_negative_pairs(std, positive_pair_set, target_neg)

train_pairs_idx = pd.concat([positive_pairs, negative_pairs], ignore_index=True)
train_pairs_idx = train_pairs_idx.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

pair_report = train_pairs_idx.groupby(["label", "source_rule"]).size().reset_index(name="n_pairs")
pair_report.to_csv(REPORTS_DIR / "candidate_pair_training_distribution.csv", index=False)
train_pairs_idx.to_parquet(PROCESSED_DIR / "ted2023_training_pair_indices.parquet", index=False)

print("Candidate-pair construction time sec:", round(time.time() - start, 1))
print("Training pair index shape:", train_pairs_idx.shape)
print(pair_report)


# 8. Materialise Pair Table and Compute Pairwise Features

This cell materialises the training pairs into a pair-level table and computes the engineered features used by the machine-learning models. It calculates character-level similarity, token-level Jaccard similarity, RapidFuzz weighted-ratio similarity, value similarity, log value difference, CPV match, country match, award-date match, lot match, contract-number match, and award-ID match. Missing textual fields are treated conservatively: missing-missing text is not interpreted as evidence of duplication. The resulting feature matrix and labels are saved for modelling.

In [ ]:
# ================================================================
# 8. MATERIALISE PAIR TABLE AND COMPUTE PAIRWISE FEATURES
# ================================================================

def make_pair_table_from_indices(df, pairs_idx):
    left = df.loc[pairs_idx["idx1"].values].reset_index(drop=True)
    right = df.loc[pairs_idx["idx2"].values].reset_index(drop=True)

    out = pd.DataFrame({
        "pair_id": [f"TED2023_TRAIN_{i:08d}" for i in range(len(pairs_idx))],
        "label": pairs_idx["label"].astype(int).values,
        "source_rule": pairs_idx["source_rule"].astype(str).values,
    })

    fields = [
        "record_id", "country", "cpv", "value_euro",
        "authority", "supplier", "title", "award_date",
        "lot", "award_id", "contract_number"
    ]

    for prefix, src in [("record1", left), ("record2", right)]:
        for f in fields:
            out[f"{prefix}_{f}"] = src[f].values

    return out


def sim_ratio(a, b):
    """
    Character-level similarity using RapidFuzz ratio.
    Missing-missing text is treated as no evidence, not as perfect similarity.
    """
    a, b = clean_text(a), clean_text(b)

    if not a or not b:
        return 0.0

    return fuzz.ratio(a, b) / 100.0


def token_jaccard(a, b):
    """
    Token-level Jaccard similarity.
    Missing-missing text is treated as no evidence, not as perfect similarity.
    """
    a_tokens = set(clean_text(a).split())
    b_tokens = set(clean_text(b).split())

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def weighted_ratio(a, b):
    """
    RapidFuzz weighted ratio similarity.
    This replaces the previous legacy 'jaro_winkler' naming.
    Missing-missing text is treated as no evidence, not as perfect similarity.
    """
    a, b = clean_text(a), clean_text(b)

    if not a or not b:
        return 0.0

    return fuzz.WRatio(a, b) / 100.0


def same_clean(a, b):
    """
    Returns 1 only when both cleaned codes are non-empty and equal.
    Empty-empty does not count as a match.
    """
    a_clean = clean_code(a)
    b_clean = clean_code(b)

    return int(a_clean != "" and a_clean == b_clean)


def same_date(a, b):
    """
    Returns 1 only when both dates are valid and equal.
    Missing-missing dates do not count as a match.
    """
    da = parse_date_safe(a)
    db = parse_date_safe(b)

    if pd.isna(da) or pd.isna(db):
        return 0

    return int(da.date() == db.date())


def value_ratio(a, b):
    """
    Bounded value similarity.
    Missing or invalid values are handled later as 0.0.
    """
    a = value_to_float(a)
    b = value_to_float(b)

    if pd.isna(a) or pd.isna(b) or a <= 0 or b <= 0:
        return np.nan

    return min(a, b) / max(a, b)


def value_abs_log_diff(a, b):
    """
    Absolute difference between log-transformed contract values.
    Missing or invalid values are handled later with finite imputation.
    """
    a = value_to_float(a)
    b = value_to_float(b)

    if pd.isna(a) or pd.isna(b) or a <= 0 or b <= 0:
        return np.nan

    return abs(np.log1p(a) - np.log1p(b))


def compute_pair_features(pair_df):
    out = pd.DataFrame(index=pair_df.index)

    field_map = {
        "cae": ("record1_authority", "record2_authority"),
        "win": ("record1_supplier", "record2_supplier"),
        "title": ("record1_title", "record2_title"),
    }

    for name, (c1, c2) in field_map.items():
        out[f"{name}_levenshtein"] = [
            sim_ratio(a, b)
            for a, b in zip(pair_df[c1], pair_df[c2])
        ]

        out[f"{name}_jaccard"] = [
            token_jaccard(a, b)
            for a, b in zip(pair_df[c1], pair_df[c2])
        ]

        out[f"{name}_weighted_ratio"] = [
            weighted_ratio(a, b)
            for a, b in zip(pair_df[c1], pair_df[c2])
        ]

    out["value_ratio"] = [
        value_ratio(a, b)
        for a, b in zip(pair_df["record1_value_euro"], pair_df["record2_value_euro"])
    ]

    out["value_abs_log_diff"] = [
        value_abs_log_diff(a, b)
        for a, b in zip(pair_df["record1_value_euro"], pair_df["record2_value_euro"])
    ]

    out["same_cpv"] = [
        same_clean(a, b)
        for a, b in zip(pair_df["record1_cpv"], pair_df["record2_cpv"])
    ]

    out["same_country"] = [
        same_clean(a, b)
        for a, b in zip(pair_df["record1_country"], pair_df["record2_country"])
    ]

    out["same_award_date"] = [
        same_date(a, b)
        for a, b in zip(pair_df["record1_award_date"], pair_df["record2_award_date"])
    ]

    out["same_lot"] = [
        same_clean(a, b)
        for a, b in zip(pair_df["record1_lot"], pair_df["record2_lot"])
    ]

    out["same_contract_number"] = [
        same_clean(a, b)
        for a, b in zip(pair_df["record1_contract_number"], pair_df["record2_contract_number"])
    ]

    out["same_award_id"] = [
        same_clean(a, b)
        for a, b in zip(pair_df["record1_award_id"], pair_df["record2_award_id"])
    ]

    out = out.replace([np.inf, -np.inf], np.nan)

    # Missing values are treated conservatively but models need finite inputs.
    out["value_ratio"] = out["value_ratio"].fillna(0.0)

    if out["value_abs_log_diff"].notna().any():
        out["value_abs_log_diff"] = out["value_abs_log_diff"].fillna(
            out["value_abs_log_diff"].median()
        )
    else:
        out["value_abs_log_diff"] = out["value_abs_log_diff"].fillna(0.0)

    out = out.fillna(0.0)

    missing_cols = [c for c in FEATURE_COLUMNS_FULL if c not in out.columns]
    if missing_cols:
        raise KeyError(
            f"Missing feature columns: {missing_cols}. "
            f"Available: {out.columns.tolist()}. "
            "Check that CONFIG['features']['similarity_features'] uses "
            "weighted_ratio names instead of jaro_winkler names."
        )

    return out[FEATURE_COLUMNS_FULL]


start = time.time()

train_pairs = make_pair_table_from_indices(std, train_pairs_idx)
X_train_all = compute_pair_features(train_pairs)
y_train_all = train_pairs["label"].astype(int).values

train_pairs.to_parquet(
    PROCESSED_DIR / "ted2023_training_pairs_materialised.parquet",
    index=False
)

X_train_all.to_csv(
    PROCESSED_DIR / "X_train_ted2023_full_source_features.csv",
    index=False
)

pd.Series(y_train_all, name="label").to_csv(
    PROCESSED_DIR / "y_train_ted2023_full_source.csv",
    index=False
)

print("Feature computation time sec:", round(time.time() - start, 1))
print("X_train_all:", X_train_all.shape)
print("y distribution:", pd.Series(y_train_all).value_counts().to_dict())

X_train_all.head()

# 9. Load External TED 2022 Gold-Standard Test Set

This cell loads the external TED 2022 annotated gold-standard file and converts it into the same pairwise feature format as the TED 2023 training data. The human annotation label is used as the final test label. Missing record fields are filled when necessary so that the same feature-engineering function can be applied. This external dataset is not used for training, tuning, threshold selection, or model selection; it is used only once for final external evaluation.

In [ ]:
# ================================================================
# 9. LOAD EXTERNAL TED 2022 GOLD-STANDARD TEST SET
# ================================================================
def load_gold_2022(path: Path):
    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    if "human_label" not in df.columns:
        raise KeyError("Gold file must contain human_label.")
    df = df[df["human_label"].isin([0, 1, "0", "1"])].copy()
    df["label"] = df["human_label"].astype(int)
    # Convert existing record column names to the names expected by compute_pair_features.
    rename_map = {
        "record1_authority": "record1_authority",
        "record2_authority": "record2_authority",
        "record1_supplier": "record1_supplier",
        "record2_supplier": "record2_supplier",
        "record1_title": "record1_title",
        "record2_title": "record2_title",
        "record1_cpv": "record1_cpv",
        "record2_cpv": "record2_cpv",
        "record1_country": "record1_country",
        "record2_country": "record2_country",
        "record1_value_euro": "record1_value_euro",
        "record2_value_euro": "record2_value_euro",
        "record1_award_date": "record1_award_date",
        "record2_award_date": "record2_award_date",
        "record1_lot": "record1_lot",
        "record2_lot": "record2_lot",
        "record1_award": "record1_award_id",
        "record2_award": "record2_award_id",
        "record1_contract_number": "record1_contract_number",
        "record2_contract_number": "record2_contract_number",
    }
    for old, new in rename_map.items():
        if old in df.columns and old != new:
            df[new] = df[old]
    required = ["record1_authority", "record2_authority", "record1_supplier", "record2_supplier", "record1_title", "record2_title", "record1_cpv", "record2_cpv", "record1_country", "record2_country", "record1_value_euro", "record2_value_euro", "record1_award_date", "record2_award_date", "record1_lot", "record2_lot", "record1_award_id", "record2_award_id", "record1_contract_number", "record2_contract_number"]
    for c in required:
        if c not in df.columns:
            df[c] = np.nan
    return df

test_pairs = load_gold_2022(ted2022_gold_path)
X_test = compute_pair_features(test_pairs)
y_test = test_pairs["label"].astype(int).values

X_test.to_csv(PROCESSED_DIR / "X_test_ted2022_gold_features.csv", index=False)
test_pairs.to_csv(PROCESSED_DIR / "ted2022_gold_pairs_used_for_external_test.csv", index=False)

annotation_audit = pd.DataFrame({
    "n_test_pairs": [len(test_pairs)],
    "n_positive": [int(np.sum(y_test == 1))],
    "n_negative": [int(np.sum(y_test == 0))],
    "positive_rate": [float(np.mean(y_test == 1))],
    "mean_human_confidence": [float(pd.to_numeric(test_pairs.get("human_confidence", pd.Series(dtype=float)), errors="coerce").mean()) if "human_confidence" in test_pairs.columns else np.nan],
    "n_low_confidence_lt_70": [int((pd.to_numeric(test_pairs.get("human_confidence", pd.Series(dtype=float)), errors="coerce") < 70).sum()) if "human_confidence" in test_pairs.columns else np.nan],
})
annotation_audit.to_csv(REPORTS_DIR / "ted2022_annotation_audit.csv", index=False)

print("External TED 2022 test shape:", X_test.shape)
print("Test label distribution:", pd.Series(y_test).value_counts().to_dict())
print(annotation_audit)


# 10. Train/Validation Split, Metrics, and Threshold Selection

This cell splits the TED 2023 constructed training pairs into an internal development set and an internal validation set using stratified sampling. It also defines the main metric functions, probability extraction, threshold-based evaluation, validation-based threshold search, and confusion-matrix plotting. The validation set is used to select decision thresholds and compare models before the final external TED 2022 evaluation.

In [ ]:
# ================================================================
# 10. TRAIN/VALIDATION SPLIT, METRICS, AND THRESHOLD SELECTION
# ================================================================
X_dev, X_val, y_dev, y_val, train_dev_pairs, val_pairs = train_test_split(
    X_train_all, y_train_all, train_pairs,
    test_size=CONFIG["models"]["validation_size"],
    stratify=y_train_all,
    random_state=RANDOM_SEED
)

print("Dev:", X_dev.shape, pd.Series(y_dev).value_counts().to_dict())
print("Validation:", X_val.shape, pd.Series(y_val).value_counts().to_dict())


def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    raise AttributeError("Model has neither predict_proba nor decision_function.")


def metrics_from_proba(y_true, proba, threshold=0.5, prefix=""):
    pred = (proba >= threshold).astype(int)
    out = {
        f"{prefix}threshold": threshold,
        f"{prefix}accuracy": accuracy_score(y_true, pred),
        f"{prefix}precision": precision_score(y_true, pred, zero_division=0),
        f"{prefix}recall": recall_score(y_true, pred, zero_division=0),
        f"{prefix}f1": f1_score(y_true, pred, zero_division=0),
        f"{prefix}pr_auc": average_precision_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan,
        f"{prefix}roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    return out


def find_best_threshold(y_true, proba):
    rows = []
    for th in CONFIG["models"]["threshold_grid"]:
        rows.append(metrics_from_proba(y_true, proba, threshold=th))
    df = pd.DataFrame(rows)
    df = df.sort_values(["f1", "pr_auc", "recall"], ascending=False).reset_index(drop=True)
    return float(df.loc[0, "threshold"]), df


def save_confusion_matrix(y_true, proba, threshold, title, filename):
    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    savefig(filename)
    return cm

# 11. Model Builders and Optuna Tuning for Ensemble Models Only

This cell defines the model-building and tuning logic. Logistic Regression is implemented as a fixed interpretable baseline with standardisation inside its pipeline. Random Forest, XGBoost, and LightGBM are treated as ensemble models and tuned with Optuna. Hyperparameter tuning is performed on a stratified subset of the internal development data for computational feasibility, while final fitting uses the full development set. GPU acceleration is attempted for XGBoost and LightGBM when available, with automatic fallback to CPU if GPU training fails.

In [ ]:
# ================================================================
# 11. MODEL BUILDERS AND OPTUNA TUNING FOR ENSEMBLE MODELS ONLY
# FINAL RUN-ALL SAFE CPU-ONLY VERSION
# ================================================================

from lightgbm.basic import LightGBMError

# ------------------------------------------------
# Force full pipeline to CPU
# ------------------------------------------------

CONFIG.setdefault("compute", {})
CONFIG["compute"]["use_gpu_if_available"] = False
CONFIG["compute"]["gpu_available"] = False

GPU_DISABLED_MODELS = {"XGBoost", "LightGBM"}


def should_use_gpu(model_name, use_gpu=None):
    """
    GPU is fully disabled in this final pipeline version.
    The function name is kept only for compatibility with later cells.
    """
    return False


def sanitize_lightgbm_params(params=None, strict=False):
    """
    Stabilises LightGBM parameters for CPU-only Run-All execution.

    The search space and fitted parameters are deliberately conservative
    because previous runs showed unstable LightGBM split failures with
    aggressive tree complexity.
    """
    p = dict(params or {})

    # Remove any device-related parameters.
    p.pop("device_type", None)
    p.pop("device", None)
    p.pop("gpu_platform_id", None)
    p.pop("gpu_device_id", None)

    if strict:
        max_depth = int(p.get("max_depth", 6))
        max_depth = max(3, min(max_depth, 8))

        num_leaves = int(p.get("num_leaves", 31))
        num_leaves = max(8, min(num_leaves, 63, (2 ** max_depth) - 1))

        n_estimators = int(p.get("n_estimators", 300))
        n_estimators = max(100, min(n_estimators, 500))

        learning_rate = float(p.get("learning_rate", 0.05))
        learning_rate = max(0.01, min(learning_rate, 0.08))

        min_child_samples = int(p.get("min_child_samples", 100))
        min_child_samples = max(80, min(min_child_samples, 200))

        reg_lambda = float(p.get("reg_lambda", 1.0))
        reg_lambda = max(1.0, min(reg_lambda, 10.0))

    else:
        max_depth = int(p.get("max_depth", 8))
        max_depth = max(3, min(max_depth, 10))

        num_leaves = int(p.get("num_leaves", 31))
        num_leaves = max(8, min(num_leaves, 96, (2 ** max_depth) - 1))

        n_estimators = int(p.get("n_estimators", 400))
        n_estimators = max(100, min(n_estimators, 700))

        learning_rate = float(p.get("learning_rate", 0.05))
        learning_rate = max(0.01, min(learning_rate, 0.12))

        min_child_samples = int(p.get("min_child_samples", 80))
        min_child_samples = max(50, min(min_child_samples, 200))

        reg_lambda = float(p.get("reg_lambda", 1.0))
        reg_lambda = max(1e-3, min(reg_lambda, 10.0))

    subsample = float(p.get("subsample", 0.8))
    subsample = max(0.60, min(subsample, 1.00))

    colsample_bytree = float(p.get("colsample_bytree", 0.8))
    colsample_bytree = max(0.60, min(colsample_bytree, 1.00))

    p.update({
        "n_estimators": n_estimators,
        "num_leaves": num_leaves,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_samples": min_child_samples,
        "reg_lambda": reg_lambda,
        "min_child_weight": 1e-3,
        "min_split_gain": 0.0,
    })

    return p


def build_model(model_name, params=None, seed=RANDOM_SEED, use_gpu=None):
    """
    Builds all models in CPU-only mode.
    """
    params = params or {}

    if model_name == "LogisticRegression":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                solver="liblinear",
                class_weight="balanced",
                random_state=seed,
                **params
            ))
        ])

    elif model_name == "RandomForest":
        model = RandomForestClassifier(
            n_jobs=-1,
            random_state=seed,
            class_weight="balanced_subsample",
            **params
        )

    elif model_name == "XGBoost":
        base_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "random_state": seed,
            "n_jobs": -1,
            "tree_method": "hist",
        }

        # GPU intentionally disabled.
        base_params.update(params)
        model = XGBClassifier(**base_params)

    elif model_name == "LightGBM":
        params = sanitize_lightgbm_params(params, strict=False)

        base_params = {
            "objective": "binary",
            "boosting_type": "gbdt",
            "random_state": seed,
            "n_jobs": -1,
            "class_weight": "balanced",
            "verbose": -1,
            "force_col_wise": True,
            "deterministic": True,
            "feature_pre_filter": False,
        }

        # GPU intentionally disabled.
        base_params.update(params)
        model = LGBMClassifier(**base_params)

    else:
        raise ValueError(model_name)

    return model


def fit_model_with_gpu_fallback(model_name, params, X, y):
    """
    CPU-only fitting function.
    The name is kept for compatibility with later cells.

    For LightGBM, if the first CPU fit fails, the model is retried once
    with stricter conservative parameters.
    """
    try:
        model = build_model(
            model_name=model_name,
            params=params,
            use_gpu=False
        )
        model.fit(X, y)
        return model

    except LightGBMError as e:
        if model_name == "LightGBM":
            print("LightGBM failed once. Retrying with stricter CPU-safe parameters.")
            print("LightGBM error:", str(e)[:500])

            safe_params = sanitize_lightgbm_params(params, strict=True)

            model = build_model(
                model_name="LightGBM",
                params=safe_params,
                use_gpu=False
            )
            model.fit(X, y)
            return model

        raise e


def make_stratified_tuning_subset(X, y, feature_cols):
    """
    Creates a stratified tuning subset from X_dev / y_dev.
    Optuna uses this subset only.
    Final model fitting still uses the full X_dev / y_dev.
    """
    n_sample = CONFIG["models"].get("tuning_sample_size", None)

    if n_sample is None or len(X) <= n_sample:
        X_tune = X[feature_cols].copy().reset_index(drop=True)
        y_tune = np.asarray(y)
        return X_tune, y_tune

    X_tune_full, _, y_tune, _ = train_test_split(
        X,
        y,
        train_size=n_sample,
        stratify=y,
        random_state=RANDOM_SEED
    )

    X_tune = X_tune_full[feature_cols].copy().reset_index(drop=True)
    y_tune = np.asarray(y_tune)

    return X_tune, y_tune


def suggest_params(trial, model_name, y_for_weight=None):
    if model_name == "LogisticRegression":
        raise ValueError(
            "LogisticRegression is a fixed baseline and is not tuned with Optuna."
        )

    if model_name == "RandomForest":
        return {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                800,
                step=100
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                4,
                30
            ),
            "min_samples_split": trial.suggest_int(
                "min_samples_split",
                2,
                20
            ),
            "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf",
                1,
                10
            ),
            "max_features": trial.suggest_categorical(
                "max_features",
                ["sqrt", "log2", None]
            ),
        }

    if model_name == "XGBoost":
        y_weight = np.asarray(y_for_weight)
        pos = max(1, np.sum(y_weight == 1))
        neg = max(1, np.sum(y_weight == 0))

        return {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                1000,
                step=100
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                2,
                10
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.25,
                log=True
            ),
            "subsample": trial.suggest_float(
                "subsample",
                0.60,
                1.00
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.60,
                1.00
            ),
            "min_child_weight": trial.suggest_float(
                "min_child_weight",
                1.0,
                20.0
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                1e-3,
                10.0,
                log=True
            ),
            "scale_pos_weight": neg / pos,
        }

    if model_name == "LightGBM":
        return {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                200,
                500,
                step=100
            ),
            "num_leaves": trial.suggest_int(
                "num_leaves",
                16,
                63
            ),
            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                8
            ),
            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.08,
                log=True
            ),
            "subsample": trial.suggest_float(
                "subsample",
                0.70,
                1.00
            ),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.70,
                1.00
            ),
            "min_child_samples": trial.suggest_int(
                "min_child_samples",
                80,
                200
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda",
                1.0,
                10.0,
                log=True
            ),
        }

    raise ValueError(model_name)


def cv_f1_score_with_gpu_fallback(model_name, params, X_use, y_use, cv):
    """
    Manual cross-validation.
    The function name is kept for compatibility, but all models run on CPU.
    """
    scores = []
    y_array = np.asarray(y_use)

    for train_idx, valid_idx in cv.split(X_use, y_array):
        X_train_fold = X_use.iloc[train_idx]
        X_valid_fold = X_use.iloc[valid_idx]
        y_train_fold = y_array[train_idx]
        y_valid_fold = y_array[valid_idx]

        model = fit_model_with_gpu_fallback(
            model_name=model_name,
            params=params,
            X=X_train_fold,
            y=y_train_fold
        )

        valid_proba = get_proba(
            model,
            X_valid_fold
        )

        valid_pred = (valid_proba >= 0.5).astype(int)

        score = f1_score(
            y_valid_fold,
            valid_pred,
            zero_division=0
        )

        scores.append(score)

    return float(np.mean(scores))


def evaluate_fixed_baseline_cv(model_name, X, y, feature_cols):
    """
    Computes CV F1 for Logistic Regression without Optuna tuning.
    This is evaluation, not hyperparameter search.
    """
    X_use = X[feature_cols]

    cv = StratifiedKFold(
        n_splits=CONFIG["models"]["cv_folds"],
        shuffle=True,
        random_state=RANDOM_SEED
    )

    model = build_model(
        model_name,
        params={},
        use_gpu=False
    )

    scores = cross_val_score(
        model,
        X_use,
        y,
        cv=cv,
        scoring="f1",
        n_jobs=1
    )

    return float(np.mean(scores))


def tune_model(model_name, X, y, feature_cols):
    if model_name not in CONFIG["models"]["tuned_models"]:
        raise ValueError(f"{model_name} is not configured for Optuna tuning.")

    X_use, y_use = make_stratified_tuning_subset(
        X=X,
        y=y,
        feature_cols=feature_cols
    )

    print(f"Optuna tuning subset for {model_name}: {X_use.shape[0]:,} rows")
    print("Tuning subset label distribution:", pd.Series(y_use).value_counts().to_dict())

    cv = StratifiedKFold(
        n_splits=CONFIG["models"]["cv_folds"],
        shuffle=True,
        random_state=RANDOM_SEED
    )

    n_trials = CONFIG["models"]["optuna_trials_by_model"][model_name]

    def objective(trial):
        params = suggest_params(
            trial=trial,
            model_name=model_name,
            y_for_weight=y_use
        )

        try:
            score = cv_f1_score_with_gpu_fallback(
                model_name=model_name,
                params=params,
                X_use=X_use,
                y_use=y_use,
                cv=cv
            )
            return score

        except (LightGBMError, ValueError, FloatingPointError) as e:
            print(f"Trial failed safely for {model_name}: {str(e)[:300]}")
            return 0.0

    sampler = optuna.samplers.TPESampler(
        seed=RANDOM_SEED
    )

    study = optuna.create_study(
        direction="maximize",
        study_name=f"{model_name}_f1",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        timeout=CONFIG["models"]["optuna_timeout_per_model_seconds"],
        show_progress_bar=True,
        catch=(LightGBMError, ValueError, FloatingPointError)
    )

    return study, X_use.shape[0]


DECLARED_MODELS = [
    "LogisticRegression",
    "RandomForest",
    "XGBoost",
    "LightGBM"
]

TUNED_MODELS = CONFIG["models"]["tuned_models"]

optuna_rows = []
validation_rows = []
best_params = {}
fitted_validation_models = {}

for model_name in DECLARED_MODELS:
    print("\n" + "=" * 80)
    print("Model:", model_name)

    start = time.time()

    if model_name in TUNED_MODELS:
        print("Optuna tuning:", model_name)

        study, tuning_n_rows = tune_model(
            model_name=model_name,
            X=X_dev,
            y=y_dev,
            feature_cols=FEATURE_COLUMNS_FULL
        )

        elapsed = time.time() - start
        params = study.best_params
        best_cv_f1 = study.best_value
        n_trials = len(study.trials)

        if model_name == "LightGBM":
            params = sanitize_lightgbm_params(
                params,
                strict=False
            )

        best_params[model_name] = params

        optuna_rows.append({
            "model": model_name,
            "best_cv_f1": best_cv_f1,
            "n_trials": n_trials,
            "tuning_n_rows": tuning_n_rows,
            "runtime_seconds": elapsed,
            "best_params_json": json.dumps(params),
        })

    else:
        print("Fixed baseline, no Optuna tuning:", model_name)

        params = {}
        tuning_n_rows = 0
        best_params[model_name] = params

        best_cv_f1 = evaluate_fixed_baseline_cv(
            model_name=model_name,
            X=X_dev,
            y=y_dev,
            feature_cols=FEATURE_COLUMNS_FULL
        )

        n_trials = 0
        elapsed = time.time() - start

    print(f"Fitting {model_name} on full X_dev / y_dev after tuning or baseline setup.")

    model = fit_model_with_gpu_fallback(
        model_name=model_name,
        params=params,
        X=X_dev[FEATURE_COLUMNS_FULL],
        y=y_dev
    )

    fitted_validation_models[model_name] = model

    val_proba = get_proba(
        model,
        X_val[FEATURE_COLUMNS_FULL]
    )

    best_th, th_df = find_best_threshold(
        y_val,
        val_proba
    )

    val_metrics_05 = metrics_from_proba(
        y_val,
        val_proba,
        threshold=0.5,
        prefix="val_05_"
    )

    val_metrics_best = metrics_from_proba(
        y_val,
        val_proba,
        threshold=best_th,
        prefix="val_best_"
    )

    validation_rows.append({
        "model": model_name,
        "tuned_with_optuna": model_name in TUNED_MODELS,
        "n_trials": n_trials,
        "tuning_n_rows": tuning_n_rows,
        "best_cv_f1": best_cv_f1,
        "best_threshold": best_th,
        **val_metrics_05,
        **val_metrics_best,
    })

    th_df.to_csv(
        REPORTS_DIR / f"threshold_sensitivity_validation_{model_name}.csv",
        index=False
    )

optuna_results = pd.DataFrame(optuna_rows).sort_values(
    "best_cv_f1",
    ascending=False
)

validation_results = pd.DataFrame(validation_rows).sort_values(
    "val_best_f1",
    ascending=False
)

optuna_results.to_csv(
    REPORTS_DIR / "optuna_results_all_models.csv",
    index=False
)

validation_results.to_csv(
    REPORTS_DIR / "validation_model_comparison_all_models.csv",
    index=False
)

print("Optuna results, tuned models only:")
try:
    display(optuna_results)
except NameError:
    print(optuna_results)

print("Validation results, all declared models:")
try:
    display(validation_results)
except NameError:
    print(validation_results)



# 12. Automatic Best Model Selection and External TED 2022 Testing

This cell selects the best model automatically based on internal validation performance and then evaluates all declared models on the external TED 2022 gold-standard test set. Each model uses its own validation-selected threshold. The best model is fixed based on validation results, not external test results. Final models are fitted on the full TED 2023 constructed training pairs, and predictions, metrics, a confusion matrix, and a model-comparison plot are saved.


In [ ]:
# ================================================================
# 12. AUTOMATIC BEST MODEL SELECTION AND EXTERNAL TED 2022 TESTING
# FINAL RUN-ALL SAFE CPU-ONLY VERSION
# ================================================================

best_model_name = validation_results.iloc[0]["model"]
best_threshold = float(validation_results.iloc[0]["best_threshold"])

print("Best model selected automatically:", best_model_name)
print("Best validation threshold:", best_threshold)

# Store each model's own validation-selected threshold.
validation_threshold_by_model = (
    validation_results
    .set_index("model")["best_threshold"]
    .astype(float)
    .to_dict()
)

# ------------------------------------------------
# Fit final versions of declared models on full TED 2023 pairs
# ------------------------------------------------

external_rows = []
final_models = {}
failed_external_models = []

for model_name in DECLARED_MODELS:
    print("\n" + "=" * 80)
    print("Final external evaluation model:", model_name)

    model_threshold = float(validation_threshold_by_model[model_name])
    print("Validation-selected threshold for this model:", model_threshold)

    try:
        model = fit_model_with_gpu_fallback(
            model_name=model_name,
            params=best_params[model_name],
            X=X_train_all[FEATURE_COLUMNS_FULL],
            y=y_train_all
        )

        final_models[model_name] = model

        test_proba = get_proba(
            model,
            X_test[FEATURE_COLUMNS_FULL]
        )

        test_pred = (test_proba >= model_threshold).astype(int)

        row = {
            "model": model_name,
            "status": "ok",
            "threshold_used": model_threshold,
            **metrics_from_proba(
                y_test,
                test_proba,
                threshold=model_threshold,
                prefix="test_"
            )
        }

        external_rows.append(row)

        prediction_df = pd.DataFrame({
            "pair_id": test_pairs.get(
                "pair_id",
                pd.Series(range(len(test_pairs)))
            ).values,
            "y_true": y_test,
            "proba": test_proba,
            "threshold_used": model_threshold,
            "pred": test_pred,
        })

        prediction_df.to_csv(
            REPORTS_DIR / f"external_test_predictions_{model_name}.csv",
            index=False
        )

    except Exception as e:
        error_msg = str(e)[:1000]
        print(f"External evaluation failed for {model_name}.")
        print("Error:", error_msg)

        failed_external_models.append({
            "model": model_name,
            "error": error_msg
        })

        # If the validation-selected final model fails, the final evaluation is invalid.
        # In that case, stop clearly.
        if model_name == best_model_name:
            raise RuntimeError(
                f"The validation-selected best model failed during final external evaluation: {model_name}"
            ) from e

        # For non-selected models, keep the notebook running and record missing metrics.
        external_rows.append({
            "model": model_name,
            "status": "failed",
            "threshold_used": model_threshold,
            "test_accuracy": np.nan,
            "test_precision": np.nan,
            "test_recall": np.nan,
            "test_f1": np.nan,
            "test_pr_auc": np.nan,
            "test_roc_auc": np.nan,
            "error": error_msg,
        })


external_results = pd.DataFrame(external_rows)

external_results = external_results.sort_values(
    "test_f1",
    ascending=False,
    na_position="last"
)

external_results.to_csv(
    REPORTS_DIR / "external_ted2022_test_model_comparison.csv",
    index=False
)

if failed_external_models:
    failed_external_models_df = pd.DataFrame(failed_external_models)
    failed_external_models_df.to_csv(
        REPORTS_DIR / "external_ted2022_failed_models.csv",
        index=False
    )
else:
    failed_external_models_df = pd.DataFrame(
        columns=["model", "error"]
    )

# ------------------------------------------------
# Keep the validation-selected best model fixed
# ------------------------------------------------

best_final_model = final_models[best_model_name]

best_test_proba = get_proba(
    best_final_model,
    X_test[FEATURE_COLUMNS_FULL]
)

best_test_pred = (best_test_proba >= best_threshold).astype(int)

print("\nExternal test results:")
try:
    display(external_results)
except NameError:
    print(external_results)

if failed_external_models:
    print("\nNon-selected models that failed during external evaluation:")
    print(failed_external_models_df)

# ------------------------------------------------
# Save confusion matrix for validation-selected best model
# ------------------------------------------------

cm = save_confusion_matrix(
    y_test,
    best_test_proba,
    best_threshold,
    f"External TED 2022 confusion matrix — {best_model_name}",
    "external_test_confusion_matrix_best_model.png"
)

# ------------------------------------------------
# Visual model comparison
# ------------------------------------------------

plot_source = external_results[
    external_results["status"].eq("ok")
].copy()

if not plot_source.empty:
    plt.figure(figsize=(10, 5))

    plot_df = plot_source.melt(
        id_vars="model",
        value_vars=[
            "test_precision",
            "test_recall",
            "test_f1",
            "test_pr_auc",
            "test_roc_auc",
        ],
        var_name="metric",
        value_name="score"
    )

    sns.barplot(
        data=plot_df,
        x="metric",
        y="score",
        hue="model"
    )

    plt.ylim(0, 1.05)
    plt.title("External TED 2022 model comparison")
    plt.xticks(rotation=30, ha="right")
    savefig("external_test_model_comparison_metrics.png")
else:
    print("No successful external model results available for plotting.")

print("\nSaved variables for later cells:")
print("best_model_name:", best_model_name)
print("best_threshold:", best_threshold)
print("best_final_model:", type(best_final_model))
print("best_test_proba shape:", np.asarray(best_test_proba).shape)
print("best_test_pred shape:", np.asarray(best_test_pred).shape)
print("external_results shape:", external_results.shape)



# 13. Bootstrap Confidence Intervals for External Test Metrics

This cell estimates the uncertainty of the selected best model’s external test performance using bootstrap resampling. It repeatedly samples the TED 2022 external test set with replacement and recalculates precision, recall, F1, PR-AUC, and ROC-AUC. Samples containing only one class are skipped because threshold-independent metrics are undefined in that case. The final output is a confidence-interval table and a visual plot of metric uncertainty.

In [ ]:
# ================================================================
# 13. BOOTSTRAP CONFIDENCE INTERVALS FOR EXTERNAL TEST METRICS
# ================================================================

def bootstrap_metric_ci(y_true, proba, threshold, n_bootstrap=500):
    rng = np.random.default_rng(RANDOM_SEED + 10)
    rows = []
    n = len(y_true)

    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    for b in range(n_bootstrap):
        idx = rng.choice(
            np.arange(n),
            size=n,
            replace=True
        )

        # Skip bootstrap samples where only one class is present.
        if len(np.unique(y_true[idx])) < 2:
            continue

        m = metrics_from_proba(
            y_true[idx],
            proba[idx],
            threshold=threshold
        )

        m["bootstrap_id"] = b
        rows.append(m)

    boot = pd.DataFrame(rows)

    if boot.empty:
        raise ValueError(
            "Bootstrap produced no valid samples. "
            "Check y_test class distribution."
        )

    ci_rows = []

    for metric in [
        "precision",
        "recall",
        "f1",
        "pr_auc",
        "roc_auc",
    ]:
        values = boot[metric].dropna()

        if values.empty:
            ci_rows.append({
                "metric": metric,
                "mean": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan,
            })
        else:
            ci_rows.append({
                "metric": metric,
                "mean": values.mean(),
                "ci_low": np.percentile(
                    values,
                    CONFIG["bootstrap"]["ci_low"]
                ),
                "ci_high": np.percentile(
                    values,
                    CONFIG["bootstrap"]["ci_high"]
                ),
            })

    return boot, pd.DataFrame(ci_rows)


boot_df, ci_df = bootstrap_metric_ci(
    y_test,
    best_test_proba,
    best_threshold,
    CONFIG["bootstrap"]["n_bootstrap"]
)

boot_df.to_csv(
    REPORTS_DIR / "bootstrap_external_test_metrics_samples.csv",
    index=False
)

ci_df.to_csv(
    REPORTS_DIR / "bootstrap_external_test_metrics_ci.csv",
    index=False
)

print(ci_df)

plt.figure(figsize=(8, 5))

plt.errorbar(
    ci_df["metric"],
    ci_df["mean"],
    yerr=[
        ci_df["mean"] - ci_df["ci_low"],
        ci_df["ci_high"] - ci_df["mean"],
    ],
    fmt="o",
    capsize=5
)

plt.ylim(0, 1.05)
plt.title(f"Bootstrap 95% CI — external TED 2022, {best_model_name}")
plt.ylabel("score")
savefig("bootstrap_ci_external_test_best_model.png")


# 14. Leakage and Feature-Set Ablation Analysis

This cell evaluates whether identifier-like features such as contract number and award ID have an excessive influence on performance. It compares a reduced feature set without identifier-like variables against the full feature set. For each feature set, the threshold is selected on the internal validation set, and final performance is measured on the external TED 2022 test set. This provides a robustness and leakage-risk check for the model.

In [ ]:
# ================================================================
# 14. LEAKAGE / FEATURE-SET ABLATION ANALYSIS
# ================================================================

ablation_configs = {
    "reduced_no_identifier_like_features": FEATURE_COLUMNS_REDUCED,
    "full_with_identifier_like_features": FEATURE_COLUMNS_FULL,
}

ablation_rows = []

for feature_set_name, cols in ablation_configs.items():
    print("\n" + "=" * 80)
    print("Running ablation:", feature_set_name)
    print("Number of features:", len(cols))

    # Validation-stage model for threshold selection.
    # Hyperparameters are kept fixed from the selected best model.
    val_model = fit_model_with_gpu_fallback(
        model_name=best_model_name,
        params=best_params[best_model_name],
        X=X_dev[cols],
        y=y_dev
    )

    val_proba = get_proba(
        val_model,
        X_val[cols]
    )

    ablation_threshold, ablation_th_df = find_best_threshold(
        y_val,
        val_proba
    )

    ablation_th_df.to_csv(
        REPORTS_DIR / f"threshold_sensitivity_ablation_{feature_set_name}.csv",
        index=False
    )

    val_metrics_best = metrics_from_proba(
        y_val,
        val_proba,
        threshold=ablation_threshold,
        prefix="val_best_"
    )

    # Final ablation model fitted on full TED 2023 training pairs.
    final_ablation_model = fit_model_with_gpu_fallback(
        model_name=best_model_name,
        params=best_params[best_model_name],
        X=X_train_all[cols],
        y=y_train_all
    )

    test_proba = get_proba(
        final_ablation_model,
        X_test[cols]
    )

    test_metrics = metrics_from_proba(
        y_test,
        test_proba,
        threshold=ablation_threshold,
        prefix="test_"
    )

    row = {
        "feature_set": feature_set_name,
        "model": best_model_name,
        "n_features": len(cols),
        "threshold_used": ablation_threshold,
        **val_metrics_best,
        **test_metrics,
    }

    ablation_rows.append(row)

    prediction_df = pd.DataFrame({
        "pair_id": test_pairs.get(
            "pair_id",
            pd.Series(range(len(test_pairs)))
        ).values,
        "y_true": y_test,
        "proba": test_proba,
        "threshold_used": ablation_threshold,
        "pred": (test_proba >= ablation_threshold).astype(int),
    })

    prediction_df.to_csv(
        REPORTS_DIR / f"ablation_external_predictions_{feature_set_name}.csv",
        index=False
    )

ablation_results = pd.DataFrame(ablation_rows).sort_values(
    "test_f1",
    ascending=False
)

ablation_results.to_csv(
    REPORTS_DIR / "leakage_feature_ablation_external_test.csv",
    index=False
)

print(ablation_results)

plt.figure(figsize=(8, 5))

plot_df = ablation_results.melt(
    id_vars="feature_set",
    value_vars=[
        "test_precision",
        "test_recall",
        "test_f1",
        "test_pr_auc",
        "test_roc_auc",
    ],
    var_name="metric",
    value_name="score"
)

sns.barplot(
    data=plot_df,
    x="metric",
    y="score",
    hue="feature_set"
)

plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.title("External test performance: reduced vs full feature set")
savefig("leakage_ablation_external_test.png")


# 15. SHAP Interpretation and Permutation-Importance Backup

This cell interprets the automatically selected best model. SHAP values are computed when supported, and mean absolute SHAP importance is exported as a feature-importance table and figure. For robustness, permutation importance is also computed using the validation-selected threshold and F1 score, ensuring consistency with the final evaluation logic. This cell supports model interpretability and explains which pairwise features contribute most to duplicate classification.

In [ ]:
# ================================================================
# 15. SHAP ON AUTOMATICALLY SELECTED BEST MODEL + PERMUTATION BACKUP
# ================================================================

def sample_rows(X, max_rows, seed):
    if len(X) <= max_rows:
        return X.copy()
    return X.sample(n=max_rows, random_state=seed)


def extract_positive_class_shap_values(shap_values):
    """
    Converts SHAP output into a 2D array with shape:
    n_rows x n_features.

    Handles:
    - list output: [class_0, class_1]
    - 2D array output
    - 3D array output from newer SHAP versions
    - SHAP Explanation objects
    """
    if isinstance(shap_values, list):
        return np.asarray(shap_values[1])

    if hasattr(shap_values, "values"):
        shap_values = shap_values.values

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 2:
        return shap_values

    if shap_values.ndim == 3:
        # Common shape: n_samples x n_features x n_classes
        if shap_values.shape[2] == 2:
            return shap_values[:, :, 1]

        # Alternative shape: n_classes x n_samples x n_features
        if shap_values.shape[0] == 2:
            return shap_values[1, :, :]

    raise ValueError(f"Unsupported SHAP output shape: {shap_values.shape}")


def selected_threshold_f1_scorer(estimator, X, y):
    """
    Permutation-importance scorer using the validation-selected threshold.
    This keeps permutation importance consistent with final test evaluation.
    """
    proba = get_proba(estimator, X)
    pred = (proba >= best_threshold).astype(int)
    return f1_score(y, pred, zero_division=0)


X_shap = sample_rows(
    X_test[FEATURE_COLUMNS_FULL],
    CONFIG["shap"]["max_explain_rows"],
    RANDOM_SEED
)

shap_values_array = None
shap_importance = None

try:
    print("Running SHAP for best model:", best_model_name)

    if best_model_name in ["RandomForest", "XGBoost", "LightGBM"]:
        explainer = shap.TreeExplainer(best_final_model)
        shap_values = explainer.shap_values(X_shap)
        shap_values_array = extract_positive_class_shap_values(shap_values)

    elif best_model_name == "LogisticRegression":
        scaler = best_final_model.named_steps["scaler"]
        clf = best_final_model.named_steps["clf"]

        X_background = sample_rows(
            X_train_all[FEATURE_COLUMNS_FULL],
            CONFIG["shap"]["max_background_rows"],
            RANDOM_SEED
        )

        X_background_scaled = scaler.transform(X_background)
        X_shap_scaled = scaler.transform(X_shap)

        explainer = shap.LinearExplainer(
            clf,
            X_background_scaled
        )

        shap_values = explainer.shap_values(X_shap_scaled)
        shap_values_array = extract_positive_class_shap_values(shap_values)

    else:
        raise ValueError(f"Unsupported model for SHAP: {best_model_name}")

    if shap_values_array.shape[1] != len(FEATURE_COLUMNS_FULL):
        raise ValueError(
            f"SHAP feature mismatch: SHAP has {shap_values_array.shape[1]} columns, "
            f"but FEATURE_COLUMNS_FULL has {len(FEATURE_COLUMNS_FULL)} columns."
        )

    shap_importance = pd.DataFrame({
        "feature": FEATURE_COLUMNS_FULL,
        "mean_abs_shap": np.abs(shap_values_array).mean(axis=0),
    }).sort_values(
        "mean_abs_shap",
        ascending=False
    )

    shap_importance.to_csv(
        REPORTS_DIR / "shap_feature_importance_best_model.csv",
        index=False
    )

    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=shap_importance.head(20),
        x="mean_abs_shap",
        y="feature"
    )
    plt.title(f"SHAP mean absolute importance — {best_model_name}")
    savefig("shap_mean_abs_importance_best_model.png")

    plt.figure()
    shap.summary_plot(
        shap_values_array,
        X_shap,
        show=False,
        max_display=20
    )
    plt.title(f"SHAP summary — {best_model_name}")
    savefig("shap_summary_best_model.png")

except Exception as e:
    print("SHAP failed, permutation importance will still be used.")
    print("SHAP error:", repr(e))


print("Running permutation importance backup with selected threshold:", best_threshold)

perm = permutation_importance(
    best_final_model,
    X_test[FEATURE_COLUMNS_FULL],
    y_test,
    scoring=selected_threshold_f1_scorer,
    n_repeats=20,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

perm_importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS_FULL,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values(
    "importance_mean",
    ascending=False
)

perm_importance.to_csv(
    REPORTS_DIR / "permutation_importance_best_model_external_test.csv",
    index=False
)

plt.figure(figsize=(8, 6))
sns.barplot(
    data=perm_importance.head(20),
    x="importance_mean",
    y="feature"
)
plt.title(f"Permutation importance backup — {best_model_name}")
savefig("permutation_importance_best_model.png")

if shap_importance is not None:
    print("Top SHAP importance:")
    print(shap_importance.head(15))
else:
    print("SHAP unavailable.")

print("Top permutation importance:")
print(perm_importance.head(15))



# 16. Detailed Error Analysis and Error Taxonomy

This cell performs detailed error analysis on the external TED 2022 test predictions of the selected best model. It labels each prediction as true positive, true negative, false positive, or false negative, attaches the engineered features, and assigns rule-based error buckets to recurring failure patterns. It also exports high-confidence errors for manual review and performs subgroup analysis by country, CPV prefix, and candidate rule. This supports robustness, ethics, and limitation analysis.

In [ ]:
# ================================================================
# 16. DETAILED ERROR ANALYSIS AND ERROR TAXONOMY
# ================================================================

error_df = test_pairs.copy().reset_index(drop=True)

error_df["y_true"] = np.asarray(y_test)
error_df["proba_duplicate"] = np.asarray(best_test_proba)
error_df["y_pred"] = np.asarray(best_test_pred)

error_df["error_type"] = np.select(
    [
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 1),
        (error_df["y_true"] == 0) & (error_df["y_pred"] == 0),
        (error_df["y_true"] == 0) & (error_df["y_pred"] == 1),
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
    ],
    [
        "TP",
        "TN",
        "FP",
        "FN",
    ],
    default="unknown",
)

# Attach engineered features used by the final model.
for c in FEATURE_COLUMNS_FULL:
    error_df[c] = X_test[c].values


def error_bucket(row):
    """
    Rule-based error taxonomy for interpretation.
    These buckets are not ground-truth labels; they describe recurring
    feature patterns among false positives and false negatives.
    """
    if row["error_type"] not in ["FP", "FN"]:
        return "correct_prediction"

    if (
        row.get("same_contract_number", 0) == 1
        or row.get("same_award_id", 0) == 1
    ):
        return "identifier_like_match_but_label_differs"

    if (
        row.get("same_lot", 0) == 0
        and row.get("same_cpv", 0) == 1
    ):
        return "same_cpv_different_lot_or_missing_lot"

    if (
        row.get("win_weighted_ratio", 0) > 0.90
        and row.get("cae_weighted_ratio", 0) > 0.90
        and row.get("value_ratio", 0) < 0.70
    ):
        return "high_name_similarity_low_value_similarity"

    if (
        row.get("title_weighted_ratio", 0) > 0.90
        and row.get("win_weighted_ratio", 0) < 0.50
    ):
        return "similar_title_different_supplier"

    if (
        row.get("value_ratio", 0) > 0.95
        and row.get("win_weighted_ratio", 0) < 0.50
    ):
        return "similar_value_different_supplier"

    return "mixed_ambiguous_case"


error_df["error_bucket"] = error_df.apply(error_bucket, axis=1)

error_summary = (
    error_df
    .groupby(["error_type", "error_bucket"])
    .size()
    .reset_index(name="n")
    .sort_values(["error_type", "n"], ascending=[True, False])
)

error_summary.to_csv(
    REPORTS_DIR / "error_taxonomy_summary.csv",
    index=False,
)

# High-confidence errors for manual inspection.
error_df["confidence"] = np.where(
    error_df["y_pred"] == 1,
    error_df["proba_duplicate"],
    1 - error_df["proba_duplicate"],
)

high_conf_errors = (
    error_df[error_df["error_type"].isin(["FP", "FN"])]
    .sort_values("confidence", ascending=False)
)

high_conf_errors.to_csv(
    REPORTS_DIR / "high_confidence_errors_external_test.csv",
    index=False,
)

high_conf_errors.head(50).to_csv(
    REPORTS_DIR / "top50_high_confidence_errors_for_manual_review.csv",
    index=False,
)


def safe_subgroup_metrics(y_true_group, proba_group, threshold):
    """
    Computes subgroup metrics safely.
    ROC-AUC and PR-AUC can be undefined when only one class is present.
    """
    try:
        return metrics_from_proba(
            y_true_group,
            proba_group,
            threshold=threshold,
            prefix="test_",
        )
    except Exception as e:
        pred_group = (proba_group >= threshold).astype(int)

        return {
            "test_accuracy": accuracy_score(y_true_group, pred_group),
            "test_precision": precision_score(
                y_true_group,
                pred_group,
                zero_division=0,
            ),
            "test_recall": recall_score(
                y_true_group,
                pred_group,
                zero_division=0,
            ),
            "test_f1": f1_score(
                y_true_group,
                pred_group,
                zero_division=0,
            ),
            "test_pr_auc": np.nan,
            "test_roc_auc": np.nan,
            "metric_warning": repr(e),
        }


# Subgroup analysis by country, CPV prefix, and candidate rule.
error_df["record1_country_clean"] = (
    error_df["record1_country"]
    .astype(str)
    .str.strip()
    .replace({"": "missing", "nan": "missing", "None": "missing"})
)

error_df["cpv1_prefix2"] = (
    error_df["record1_cpv"]
    .astype(str)
    .str.strip()
    .str[:2]
    .replace({"": "missing", "na": "missing", "nan": "missing", "No": "missing"})
)

if "candidate_rule" not in error_df.columns and "source_rule" in error_df.columns:
    error_df["candidate_rule"] = error_df["source_rule"]

subgroup_rows = []

for group_col in [
    "record1_country_clean",
    "cpv1_prefix2",
    "candidate_rule",
]:
    if group_col not in error_df.columns:
        continue

    for group, g in error_df.groupby(group_col, dropna=False):
        if len(g) < 10:
            continue

        proba_g = g["proba_duplicate"].values
        y_g = g["y_true"].values

        metrics_g = safe_subgroup_metrics(
            y_g,
            proba_g,
            threshold=best_threshold,
        )

        subgroup_rows.append({
            "group_type": group_col,
            "group": group,
            "n": len(g),
            "positive_rate": float(np.mean(y_g)),
            "fp_count": int((g["error_type"] == "FP").sum()),
            "fn_count": int((g["error_type"] == "FN").sum()),
            **metrics_g,
        })

subgroup_results = pd.DataFrame(subgroup_rows)

if not subgroup_results.empty:
    subgroup_results = subgroup_results.sort_values(
        ["group_type", "n"],
        ascending=[True, False],
    )

subgroup_results.to_csv(
    REPORTS_DIR / "subgroup_performance_external_test.csv",
    index=False,
)

print("Error taxonomy:")
print(error_summary)

print("High-confidence errors:", high_conf_errors.shape)

print("Subgroup results:")
print(subgroup_results.head(30))


plt.figure(figsize=(8, 5))

sns.countplot(
    data=error_df,
    x="error_type",
    order=["TP", "TN", "FP", "FN"],
)

plt.title("External TED 2022 error types")
plt.xlabel("Error type")
plt.ylabel("Count")
savefig("external_test_error_type_counts.png")


error_only_df = error_df[error_df["error_type"].isin(["FP", "FN"])].copy()

if not error_only_df.empty:
    bucket_order = (
        error_only_df["error_bucket"]
        .value_counts()
        .index
    )

    plt.figure(figsize=(10, 5))

    sns.countplot(
        data=error_only_df,
        y="error_bucket",
        order=bucket_order,
    )

    plt.title("External TED 2022 error taxonomy")
    plt.xlabel("Count")
    plt.ylabel("Error bucket")
    savefig("external_test_error_taxonomy.png")

else:
    print("No FP/FN errors found; error taxonomy plot was skipped.")



# 17. RQ2 Pair-Level CTGAN and TVAE Synthetic Data Generation

This cell generates synthetic candidate-pair feature-table data for RQ2 using CTGAN and TVAE. Synthetic generation is based only on the internal TED 2023 training split, while the validation split remains reserved for threshold selection and model selection. The target label and binary match indicators are treated as categorical variables. Generated synthetic features are cleaned, bounded, and saved. If a synthesizer generates only one label class, this is reported rather than artificially corrected.

In [ ]:
# ================================================================
# 17. RQ2: PAIR-LEVEL CTGAN / TVAE SYNTHETIC DATA GENERATION
# FINAL RUN-ALL SAFE CPU-ONLY VERSION
# ================================================================

if CONFIG["synthetic"]["enabled"]:

    # Use only the internal TED 2023 training split as the source for synthetic generation.
    # The validation split remains reserved for threshold selection and model selection.
    if "X_dev" not in globals() or "y_dev" not in globals():
        raise RuntimeError(
            "X_dev/y_dev are required for RQ2 synthetic generation. "
            "Synthetic data must be generated from the internal TED 2023 training split only."
        )

    synthetic_source_X = X_dev[FEATURE_COLUMNS_FULL].copy()
    synthetic_source_y = np.asarray(y_dev).astype(int)
    synthetic_source_name = "TED 2023 internal training split only"

    synthetic_train_table = synthetic_source_X.copy()
    synthetic_train_table["label"] = synthetic_source_y

    synthetic_train_table.to_csv(
        PROCESSED_DIR / "synthetic_training_source_pair_feature_table.csv",
        index=False
    )

    n_synth = CONFIG["synthetic"]["num_synthetic_rows"] or len(synthetic_train_table)

    print("Synthetic source:", synthetic_source_name)
    print("Synthetic source table shape:", synthetic_train_table.shape)
    print("Synthetic rows to generate per model:", n_synth)

    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(data=synthetic_train_table)

    # Treat the target and binary indicator features as categorical.
    try:
        metadata.update_column(column_name="label", sdtype="categorical")
    except Exception:
        pass

    for c in FEATURE_COLUMNS_FULL:
        if c.startswith("same_"):
            try:
                metadata.update_column(column_name=c, sdtype="categorical")
            except Exception:
                pass

    synth_outputs = {}
    synth_quality_rows = []

    for synth_name, SynthClass, epochs in [
        ("CTGAN", CTGANSynthesizer, CONFIG["synthetic"]["ctgan_epochs"]),
        ("TVAE", TVAESynthesizer, CONFIG["synthetic"]["tvae_epochs"]),
    ]:
        print("\n" + "=" * 80)
        print("Training synthesizer:", synth_name)
        print("Epochs:", epochs)
        print("Device: CPU only")

        start = time.time()

        try:
            synthesizer = SynthClass(
                metadata,
                epochs=epochs,
                verbose=True,
                cuda=False
            )
        except TypeError:
            # Compatibility fallback for SDV versions that do not expose cuda.
            synthesizer = SynthClass(
                metadata,
                epochs=epochs,
                verbose=True
            )

        synthesizer.fit(synthetic_train_table)

        synth = synthesizer.sample(num_rows=n_synth)

        # Clean generated feature output.
        for c in FEATURE_COLUMNS_FULL:
            synth[c] = pd.to_numeric(
                synth[c],
                errors="coerce"
            ).fillna(0.0)

            if c in CONFIG["features"]["similarity_features"] or c == "value_ratio":
                synth[c] = synth[c].clip(0, 1)

            if c.startswith("same_"):
                synth[c] = (synth[c] >= 0.5).astype(int)

        # Clean generated labels, but do not randomly replace collapsed labels.
        synth["label"] = (
            pd.to_numeric(synth["label"], errors="coerce")
            .round()
            .clip(0, 1)
            .fillna(0)
            .astype(int)
        )

        label_distribution = synth["label"].value_counts().to_dict()
        label_nunique = synth["label"].nunique()

        if label_nunique < 2:
            print(
                f"WARNING: {synth_name} generated only one label class. "
                "This will be reported and handled in utility evaluation."
            )

        synth.to_csv(
            PROCESSED_DIR / f"synthetic_pairs_{synth_name.lower()}.csv",
            index=False
        )

        synth_outputs[synth_name] = synth

        synth_quality_rows.append({
            "synthetic_model": synth_name,
            "source_table": synthetic_source_name,
            "n_source_rows": len(synthetic_train_table),
            "n_synthetic_rows": len(synth),
            "epochs": epochs,
            "label_nunique": label_nunique,
            "label_distribution": str(label_distribution),
            "time_seconds": round(time.time() - start, 1),
        })

        print(synth_name, "done.")
        print("Shape:", synth.shape)
        print("Time sec:", round(time.time() - start, 1))
        print("Label distribution:", label_distribution)

    synth_quality = pd.DataFrame(synth_quality_rows)

    synth_quality.to_csv(
        REPORTS_DIR / "rq2_synthetic_generation_quality.csv",
        index=False
    )

    print("Synthetic generation quality:")
    print(synth_quality)

else:
    synth_outputs = {}
    synthetic_train_table = None
    synth_quality = pd.DataFrame()
    print("Synthetic stage disabled.")




# 18. RQ2 Utility Evaluation and Memorisation-Risk Audit

This cell evaluates the utility and memorisation risk of the synthetic pair-level data. It compares real-only training, synthetic-only training, and real-plus-synthetic training scenarios using the same selected model family. Each scenario selects its decision threshold on the real internal validation set and is evaluated once on the external TED 2022 test set. The memorisation audit checks exact feature matches, exact feature-plus-label matches, and nearest-neighbour similarity between synthetic and real training rows.

In [ ]:
# ================================================================
# 18. RQ2 UTILITY EVALUATION AND MEMORISATION-RISK AUDIT
# ================================================================

def select_threshold_for_scenario(model, X_validation, y_validation):
    """
    Selects the decision threshold on the internal real validation set.
    External TED 2022 is not used for threshold selection.
    """
    val_proba = get_proba(
        model,
        X_validation[FEATURE_COLUMNS_FULL]
    )

    threshold, threshold_df = find_best_threshold(
        y_validation,
        val_proba
    )

    return float(threshold), threshold_df


def fit_and_eval_scenario(name, X_s, y_s):
    """
    Fits the selected best model family under one RQ2 training scenario,
    selects the threshold on the internal TED 2023 validation set,
    and evaluates once on the external TED 2022 test set.
    """
    y_s = np.asarray(y_s).astype(int)

    row_base = {
        "scenario": name,
        "model_family": best_model_name,
        "n_train_rows": len(X_s),
        "train_positive_rate": float(np.mean(y_s)) if len(y_s) else np.nan,
    }

    if len(np.unique(y_s)) < 2:
        print(
            f"WARNING: Scenario '{name}' has only one label class. "
            "Utility metrics are set to NaN."
        )

        return {
            **row_base,
            "threshold_used": np.nan,
            "test_accuracy": np.nan,
            "test_precision": np.nan,
            "test_recall": np.nan,
            "test_f1": np.nan,
            "test_pr_auc": np.nan,
            "test_roc_auc": np.nan,
            "scenario_warning": "single_class_training_data",
        }

    model = fit_model_with_gpu_fallback(
        model_name=best_model_name,
        params=best_params[best_model_name],
        X=X_s[FEATURE_COLUMNS_FULL],
        y=y_s
    )

    if "X_val" in globals() and "y_val" in globals():
        scenario_threshold, threshold_df = select_threshold_for_scenario(
            model,
            X_val,
            y_val
        )

        safe_name = (
            name.lower()
            .replace(" ", "_")
            .replace("+", "plus")
            .replace("/", "_")
        )

        threshold_df.to_csv(
            REPORTS_DIR / f"rq2_threshold_sensitivity_{safe_name}.csv",
            index=False
        )

    else:
        scenario_threshold = best_threshold
        print(
            "WARNING: X_val/y_val not available. "
            "Using best_threshold fallback:",
            scenario_threshold
        )

    test_proba = get_proba(
        model,
        X_test[FEATURE_COLUMNS_FULL]
    )

    test_metrics = metrics_from_proba(
        y_test,
        test_proba,
        threshold=scenario_threshold,
        prefix="test_"
    )

    return {
        **row_base,
        "threshold_used": scenario_threshold,
        **test_metrics,
        "scenario_warning": "",
    }


utility_rows = []

# Real-only scenario uses the internal training split if available.
if "X_dev" in globals() and "y_dev" in globals():
    real_utility_X = X_dev[FEATURE_COLUMNS_FULL].copy()
    real_utility_y = np.asarray(y_dev).astype(int)
else:
    real_utility_X = X_train_all[FEATURE_COLUMNS_FULL].copy()
    real_utility_y = np.asarray(y_train_all).astype(int)

utility_rows.append(
    fit_and_eval_scenario(
        "Constructed TED 2023 real pair data only",
        real_utility_X,
        real_utility_y
    )
)

for synth_name, synth in synth_outputs.items():
    X_synth = synth[FEATURE_COLUMNS_FULL].copy()
    y_synth = synth["label"].astype(int).values

    utility_rows.append(
        fit_and_eval_scenario(
            f"{synth_name} synthetic pair data only",
            X_synth,
            y_synth
        )
    )

    X_aug = pd.concat(
        [
            real_utility_X[FEATURE_COLUMNS_FULL],
            X_synth[FEATURE_COLUMNS_FULL],
        ],
        ignore_index=True
    )

    y_aug = np.concatenate(
        [
            real_utility_y,
            y_synth,
        ]
    )

    utility_rows.append(
        fit_and_eval_scenario(
            f"Constructed TED 2023 + {synth_name} synthetic pair data",
            X_aug,
            y_aug
        )
    )

rq2_utility = pd.DataFrame(utility_rows).sort_values(
    "test_f1",
    ascending=False,
    na_position="last"
)

rq2_utility.to_csv(
    REPORTS_DIR / "rq2_synthetic_utility_external_test.csv",
    index=False
)

print("RQ2 utility:")
print(rq2_utility)

plt.figure(figsize=(11, 5))

plot_df = rq2_utility.melt(
    id_vars="scenario",
    value_vars=[
        "test_precision",
        "test_recall",
        "test_f1",
        "test_pr_auc",
        "test_roc_auc",
    ],
    var_name="metric",
    value_name="score"
)

sns.barplot(
    data=plot_df,
    x="metric",
    y="score",
    hue="scenario"
)

plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.title("RQ2 utility on fixed external TED 2022 test set")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
savefig("rq2_synthetic_utility_external_test.png")


def hash_exact_match_rate(real_df, synth_df, cols):
    real_rounded = real_df[cols].copy().round(6)
    synth_rounded = synth_df[cols].copy().round(6)

    real_hashes = set(
        pd.util.hash_pandas_object(
            real_rounded,
            index=False
        ).astype(str)
    )

    synth_hashes = pd.util.hash_pandas_object(
        synth_rounded,
        index=False
    ).astype(str)

    exact_count = int(synth_hashes.isin(real_hashes).sum())
    exact_rate = float(synth_hashes.isin(real_hashes).mean())

    return exact_count, exact_rate


def memorisation_audit(real_table, synth_table, name):
    """
    Memorisation-risk audit based on:
    1. exact feature-only matches,
    2. exact feature-plus-label matches,
    3. nearest-neighbour similarity in scaled feature space.
    """
    real_features = real_table[FEATURE_COLUMNS_FULL].copy()
    synth_features = synth_table[FEATURE_COLUMNS_FULL].copy()

    real_features = real_features.apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(0.0)

    synth_features = synth_features.apply(
        pd.to_numeric,
        errors="coerce"
    ).fillna(0.0)

    real_for_hash = real_features.copy()
    synth_for_hash = synth_features.copy()

    real_for_hash["label"] = real_table["label"].astype(int).values
    synth_for_hash["label"] = synth_table["label"].astype(int).values

    feature_exact_count, feature_exact_rate = hash_exact_match_rate(
        real_for_hash,
        synth_for_hash,
        FEATURE_COLUMNS_FULL
    )

    full_exact_count, full_exact_rate = hash_exact_match_rate(
        real_for_hash,
        synth_for_hash,
        FEATURE_COLUMNS_FULL + ["label"]
    )

    scaler = StandardScaler()

    real_scaled = scaler.fit_transform(real_features)
    synth_scaled = scaler.transform(synth_features)

    nn = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean"
    )

    nn.fit(real_scaled)

    distances, indices = nn.kneighbors(synth_scaled)

    distances = distances.ravel()
    indices = indices.ravel()

    nn_similarity = 1 / (1 + distances)

    rows = {
        "synthetic_model": name,
        "n_real_source_rows": len(real_features),
        "n_synthetic_rows": len(synth_features),
        "feature_exact_match_count": feature_exact_count,
        "feature_exact_match_rate": feature_exact_rate,
        "feature_plus_label_exact_match_count": full_exact_count,
        "feature_plus_label_exact_match_rate": full_exact_rate,
        "nn_similarity_mean": float(np.mean(nn_similarity)),
        "nn_similarity_median": float(np.median(nn_similarity)),
        "nn_similarity_p95": float(np.percentile(nn_similarity, 95)),
        "nn_similarity_p99": float(np.percentile(nn_similarity, 99)),
    }

    for th in CONFIG["synthetic"]["memorisation_nn_thresholds"]:
        rows[f"nn_similarity_ge_{th}"] = int(
            np.sum(nn_similarity >= th)
        )

        rows[f"nn_similarity_ge_{th}_rate"] = float(
            np.mean(nn_similarity >= th)
        )

    detail = synth_table.copy()
    detail["nearest_real_row_index"] = indices
    detail["nearest_distance"] = distances
    detail["nearest_similarity"] = nn_similarity

    detail.to_csv(
        REPORTS_DIR / f"memorisation_detail_{name.lower()}.csv",
        index=False
    )

    return rows


mem_rows = []

if CONFIG["synthetic"]["enabled"] and synthetic_train_table is not None:
    real_source_table = synthetic_train_table.copy()
else:
    real_source_table = X_train_all[FEATURE_COLUMNS_FULL].copy()
    real_source_table["label"] = y_train_all.astype(int)

for synth_name, synth in synth_outputs.items():
    mem_rows.append(
        memorisation_audit(
            real_source_table,
            synth,
            synth_name
        )
    )

memorisation_results = pd.DataFrame(mem_rows)

memorisation_results.to_csv(
    REPORTS_DIR / "rq2_memorisation_risk_audit.csv",
    index=False
)

print("Memorisation-risk audit:")
print(memorisation_results)

if len(memorisation_results):
    risk_value_cols = [
        "feature_exact_match_rate",
        "feature_plus_label_exact_match_rate",
    ]

    for th in CONFIG["synthetic"]["memorisation_nn_thresholds"]:
        col = f"nn_similarity_ge_{th}_rate"
        if col in memorisation_results.columns:
            risk_value_cols.append(col)

    plt.figure(figsize=(8, 5))

    mem_plot = memorisation_results.melt(
        id_vars="synthetic_model",
        value_vars=risk_value_cols,
        var_name="risk_metric",
        value_name="rate"
    )

    sns.barplot(
        data=mem_plot,
        x="risk_metric",
        y="rate",
        hue="synthetic_model"
    )

    plt.xticks(rotation=30, ha="right")
    plt.title("RQ2 memorisation-risk audit")
    savefig("rq2_memorisation_risk_audit.png")


# 19. Final DSS Report Export, Manifest, Audit Checklist, and ZIP Package

This cell exports the final DSS pipeline outputs. It saves the selected best model, model metadata, thesis-ready Excel tables, a run manifest, a DSS/Sprint audit checklist, a compact Markdown run report, and a final ZIP package containing reports, figures, and saved models. The manifest records the data sizes, configuration, selected model, selected threshold, feature sets, environment information, and output paths. This cell supports reproducibility, thesis writing, and final project delivery.


In [ ]:
# ================================================================
# 19. DSS FINAL REPORT EXPORT: TABLES, MANIFEST, AUDIT CHECKLIST, ZIP
# ================================================================

import sys
import json
import platform
import shutil
import joblib

# ----------------------------------------------------------------
# Save best final model
# ----------------------------------------------------------------

best_model_path = MODELS_DIR / f"best_model_{best_model_name}_{RUN_ID}.joblib"

joblib.dump(
    best_final_model,
    best_model_path
)

model_metadata = {
    "run_id": RUN_ID,
    "best_model": best_model_name,
    "best_threshold": best_threshold,
    "feature_columns_full": FEATURE_COLUMNS_FULL,
    "feature_columns_reduced": FEATURE_COLUMNS_REDUCED,
    "model_path": str(best_model_path),
}

with open(MODELS_DIR / f"best_model_metadata_{RUN_ID}.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2, default=str)


# ----------------------------------------------------------------
# Helper functions for safe report export
# ----------------------------------------------------------------

def get_df_if_exists(name):
    """
    Returns a DataFrame if the variable exists.
    Otherwise returns an empty DataFrame with a warning column.
    """
    if name in globals():
        obj = globals()[name]

        if obj is None:
            return pd.DataFrame({"warning": [f"{name} is None"]})

        if isinstance(obj, pd.DataFrame):
            if obj.empty:
                return pd.DataFrame({"warning": [f"{name} is empty"]})
            return obj

        if isinstance(obj, dict):
            return pd.DataFrame([obj])

        return pd.DataFrame({"warning": [f"{name} exists but is not a DataFrame"]})

    return pd.DataFrame({"warning": [f"{name} was not found in globals"]})


def write_sheet(writer, df, sheet_name):
    """
    Writes a DataFrame to Excel using a valid Excel sheet name.
    Excel sheet names must be <= 31 characters.
    """
    safe_sheet_name = sheet_name[:31]
    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)


# ----------------------------------------------------------------
# Excel report with thesis-ready tables
# ----------------------------------------------------------------

excel_path = REPORTS_DIR / f"deniz_dss_final_results_tables_{RUN_ID}.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    write_sheet(writer, pd.DataFrame([eda_summary]), "eda_summary")
    write_sheet(writer, get_df_if_exists("missing"), "missing_values")
    write_sheet(writer, get_df_if_exists("dup_risk"), "duplicate_risk")
    write_sheet(writer, get_df_if_exists("pair_report"), "train_pair_distribution")
    write_sheet(writer, get_df_if_exists("annotation_audit"), "ted2022_annotation")

    write_sheet(writer, get_df_if_exists("optuna_results"), "optuna_tuned_models")
    write_sheet(writer, get_df_if_exists("validation_results"), "validation_models")
    write_sheet(writer, get_df_if_exists("external_results"), "external_test_models")
    write_sheet(writer, get_df_if_exists("ci_df"), "bootstrap_ci")
    write_sheet(writer, get_df_if_exists("ablation_results"), "feature_ablation")

    if "shap_importance" in globals() and shap_importance is not None:
        write_sheet(writer, shap_importance, "shap_importance")
    else:
        write_sheet(
            writer,
            pd.DataFrame({"warning": ["SHAP was unavailable or failed. See permutation importance backup."]}),
            "shap_importance"
        )

    write_sheet(writer, get_df_if_exists("perm_importance"), "permutation_importance")
    write_sheet(writer, get_df_if_exists("error_summary"), "error_taxonomy")
    write_sheet(writer, get_df_if_exists("subgroup_results"), "subgroup_performance")

    if "synth_quality" in globals():
        write_sheet(writer, get_df_if_exists("synth_quality"), "rq2_synth_quality")

    write_sheet(writer, get_df_if_exists("rq2_utility"), "rq2_utility")
    write_sheet(writer, get_df_if_exists("memorisation_results"), "rq2_memorisation")


# ----------------------------------------------------------------
# Run manifest
# ----------------------------------------------------------------

manifest = {
    "run_id": RUN_ID,
    "project_root": str(PROJECT_ROOT),
    "ted2023_csv": str(ted2023_csv),
    "ted2022_gold_path": str(ted2022_gold_path),
    "python": sys.version,
    "platform": platform.platform(),
    "config": CONFIG,
    "full_ted2023_records": int(len(std)),
    "training_pairs": int(len(train_pairs)),
    "training_positive_pairs": int(np.sum(y_train_all == 1)),
    "training_negative_pairs": int(np.sum(y_train_all == 0)),
    "external_test_pairs": int(len(test_pairs)),
    "external_test_positive_pairs": int(np.sum(y_test == 1)),
    "external_test_negative_pairs": int(np.sum(y_test == 0)),
    "declared_models": DECLARED_MODELS,
    "fixed_baseline_model": "LogisticRegression",
    "optuna_tuned_models": CONFIG["models"]["tuned_models"],
    "best_model": best_model_name,
    "best_threshold": best_threshold,
    "feature_columns_full": FEATURE_COLUMNS_FULL,
    "feature_columns_reduced": FEATURE_COLUMNS_REDUCED,
    "best_model_path": str(best_model_path),
    "outputs_excel": str(excel_path),
}

manifest_path = REPORTS_DIR / f"run_manifest_{RUN_ID}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)


# ----------------------------------------------------------------
# DSS / Sprint audit mapping
# ----------------------------------------------------------------

audit_items = [
    (
        "1",
        "Full TED 2023 source ingestion",
        "PASS",
        "eda_summary.json; ted2023_full_standardised.parquet",
    ),
    (
        "2",
        "EDA: missingness, distributions, values, dates, and duplicate-risk indicators",
        "PASS",
        "eda_*.csv; eda_*.png",
    ),
    (
        "3",
        "Candidate-pair construction report from full TED 2023 source population",
        "PASS",
        "candidate_positive_rule_report.csv; candidate_pair_training_distribution.csv",
    ),
    (
        "4",
        "External TED 2022 gold-standard used only for final external evaluation",
        "PASS",
        "ted2022_annotation_audit.csv",
    ),
    (
        "5",
        "Fixed interpretable baseline and ensemble models compared",
        "PASS",
        "validation_model_comparison_all_models.csv; external_ted2022_test_model_comparison.csv",
    ),
    (
        "6",
        "Logistic Regression kept as fixed baseline; Optuna tuning applied only to Random Forest, XGBoost, and LightGBM",
        "PASS",
        "optuna_results_all_models.csv; CONFIG['models']['tuned_models']",
    ),
    (
        "7",
        "Automatic best-model selection based on internal validation performance",
        "PASS",
        "validation_model_comparison_all_models.csv",
    ),
    (
        "8",
        "Validation-based threshold selection before external testing",
        "PASS",
        "threshold_sensitivity_validation_*.csv",
    ),
    (
        "9",
        "Final external evaluation on held-out TED 2022 annotated gold-standard",
        "PASS",
        "external_ted2022_test_model_comparison.csv",
    ),
    (
        "10",
        "SHAP interpretability on selected best model",
        "PASS",
        "shap_feature_importance_best_model.csv; shap_*.png",
    ),
    (
        "11",
        "Permutation importance backup for interpretability",
        "PASS",
        "permutation_importance_best_model_external_test.csv",
    ),
    (
        "12",
        "Identifier-like feature ablation for leakage/robustness assessment",
        "PASS",
        "leakage_feature_ablation_external_test.csv",
    ),
    (
        "13",
        "Bootstrap confidence intervals for external test metrics",
        "PASS",
        "bootstrap_external_test_metrics_ci.csv",
    ),
    (
        "14",
        "Detailed error taxonomy and high-confidence error review",
        "PASS",
        "error_taxonomy_summary.csv; high_confidence_errors_external_test.csv",
    ),
    (
        "15",
        "Subgroup analysis by country, CPV prefix, and candidate rule",
        "PASS",
        "subgroup_performance_external_test.csv",
    ),
    (
        "16",
        "RQ2 utility evaluation: real-only, synthetic-only, and real-plus-synthetic training scenarios",
        "PASS",
        "rq2_synthetic_utility_external_test.csv",
    ),
    (
        "17",
        "RQ2 memorisation-risk audit using exact-match and nearest-neighbour checks",
        "PASS",
        "rq2_memorisation_risk_audit.csv",
    ),
    (
        "18",
        "Thesis-ready Excel tables exported",
        "PASS",
        excel_path.name,
    ),
    (
        "19",
        "Run manifest with data sizes, parameters, model choice, threshold, and environment",
        "PASS",
        manifest_path.name,
    ),
    (
        "20",
        "Reproducibility supported through fixed seed, environment specification, and saved model artifact",
        "PASS",
        "CONFIG; environment.yml; requirements.txt; best_model_*.joblib",
    ),
]

audit_df = pd.DataFrame(
    audit_items,
    columns=[
        "item",
        "requirement",
        "status",
        "evidence",
    ],
)

audit_path = REPORTS_DIR / "dss_rubric_sprint_audit.csv"

audit_df.to_csv(
    audit_path,
    index=False,
)


# ----------------------------------------------------------------
# Compact markdown report for quick thesis writing
# ----------------------------------------------------------------

report_md = f"""
# DSS Final Pipeline Run Report

Run ID: `{RUN_ID}`

## Data
- Full TED 2023 source records: {len(std):,}
- Constructed TED 2023 training candidate pairs: {len(train_pairs):,}
- Training positive pairs: {int(np.sum(y_train_all == 1)):,}
- Training negative pairs: {int(np.sum(y_train_all == 0)):,}
- External TED 2022 gold-standard test pairs: {len(test_pairs):,}

## Model Selection
- Fixed baseline model: Logistic Regression
- Optuna-tuned ensemble models: Random Forest, XGBoost, LightGBM
- Automatically selected best model: **{best_model_name}**
- Validation-selected threshold: **{best_threshold:.2f}**

## Key Output Files
- Excel tables: `{excel_path.name}`
- Model comparison: `external_ted2022_test_model_comparison.csv`
- Bootstrap confidence intervals: `bootstrap_external_test_metrics_ci.csv`
- SHAP: `shap_feature_importance_best_model.csv`
- Permutation importance: `permutation_importance_best_model_external_test.csv`
- Error taxonomy: `error_taxonomy_summary.csv`
- Subgroup performance: `subgroup_performance_external_test.csv`
- RQ2 utility: `rq2_synthetic_utility_external_test.csv`
- RQ2 memorisation: `rq2_memorisation_risk_audit.csv`
- Audit checklist: `dss_rubric_sprint_audit.csv`
- Saved best model: `{best_model_path.name}`
"""

report_md_path = REPORTS_DIR / f"dss_final_run_report_{RUN_ID}.md"

report_md_path.write_text(
    report_md,
    encoding="utf-8",
)


# ----------------------------------------------------------------
# Zip compact output package: reports + figures + saved models
# ----------------------------------------------------------------

package_dir = PROJECT_ROOT / f"deniz_dss_final_package_{RUN_ID}"

if package_dir.exists():
    shutil.rmtree(package_dir)

package_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for folder_name, folder_path in [
    ("reports", REPORTS_DIR),
    ("figures", FIGURES_DIR),
    ("models", MODELS_DIR),
]:
    target = package_dir / folder_name
    target.mkdir(
        parents=True,
        exist_ok=True,
    )

    for item in folder_path.glob("*"):
        if item.is_file():
            shutil.copy2(
                item,
                target / item.name,
            )

zip_base = PROJECT_ROOT / f"deniz_dss_final_outputs_{RUN_ID}"

if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=package_dir,
)

print("Best model saved:", best_model_path)
print("Excel report:", excel_path)
print("Run manifest:", manifest_path)
print("Audit checklist:", audit_path)
print("Markdown report:", report_md_path)
print("Output ZIP:", zip_base.with_suffix(".zip"))
print("DSS-final pipeline completed.")